# 📗 합성곱 신경망: 손글씨 숫자를 맞히는 CNN 만들기

CNN(합성곱 신경망, convolutional neural network)은 3 × 3 같은 작은 필터로 그림을 훑으며 이웃 픽셀을 묶어 선·모서리 같은 특징을 찾는 신경망입니다. 이 교안은 합성곱과 풀링을 작은 숫자 그림으로 직접 계산해 본 뒤, 둘을 쌓은 CNN과 깊은 CNN의 부품(배치 정규화, 잔차 블록)을 만들고, CNN으로 손글씨 숫자를 분류해 학습한 모델을 저장합니다. 지난 시간 DNN과 같은 데이터와 설정으로 학습해 결과를 비교하고, 과적합 여부는 교안 02에서 검증 데이터로 가립니다.

<img src="images/60_lesson01_goals.png" width="1050" alt="제목은 '이번 교안에서 배울 것: 손글씨 숫자를 맞히는 CNN 만들기'이고, 제목 아래 띠에 'MNIST 손글씨 숫자 학습용 10,000장, 테스트용 2,000장'이 적혀 있습니다. 판 네 개가 2행 2열로 놓여 왼쪽 위 1, 오른쪽 위 2, 왼쪽 아래 3, 오른쪽 아래 4 순서로 화살표로 이어집니다. 1 '합성곱'(칩 '3 x 3 필터를 곱해서 더하기') 판에는 가로 5칸, 세로 5칸인 '5 x 5' 그림의 왼쪽 위 3 x 3 구역에 빨간 테두리가 있고 옆에 '3 x 3 필터'가 있습니다. 2 '풀링과 CNN 쌓기'(칩 '채널, 풀링, 배치 정규화, 잔차 블록') 판에는 점점 작아지는 상자 다섯 개가 '(64, 1, 28, 28)'에서 '(64, 10)'으로 이어지고, 둘째와 셋째 상자 위로 지름길 곡선이 건너가 '+' 원에 닿습니다. 3 '학습과 결과 읽기'(칩 '필터, 특징 맵, 틀린 그림') 판에는 흑백 '필터', 손글씨 7의 '특징 맵', 빨간 X가 붙은 '틀린 그림'이 있습니다. 4 '저장과 불러오기'(칩 '같은 예측 확인') 판에는 '모델'에서 'state_dict' 문서를 거쳐 '새 모델'로 가는 화살표가 있고, 개발자(초록 옷)가 새 모델을 가리킵니다.">


**학습 목표** (그림의 번호와 같습니다)

1. **그림을 (그림 수, 채널, 높이, 너비) 텐서로 바꾸고, 3 × 3 필터의 합성곱을 손으로 계산해 `nn.Conv2d` 결과와 맞춥니다.**
2. **padding·채널·최댓값 풀링이 출력 모양을 어떻게 바꾸는지 계산해 CNN을 `nn.Sequential`로 쌓고, 깊은 CNN의 부품인 배치 정규화와 잔차 블록(ResNet)을 설명합니다.**
3. **지난 시간의 학습 함수로 CNN을 학습하고, 학습한 필터·특징 맵·틀린 그림으로 결과를 읽습니다.**
4. **`state_dict`로 학습한 모델을 저장하고 불러와 같은 예측을 내는지 확인합니다.**

## 1. 실습 준비

지난 시간의 DNN은 그림을 784개 한 줄로 펼쳐, 바로 위아래로 붙은 0행 5열과 1행 5열 픽셀이 5번과 33번 자리로 28칸 떨어졌습니다. CNN은 이웃 관계를 지키려고 그림을 가로·세로 모양 그대로 받으므로, 데이터를 불러와 CNN이 받는 모양으로 바꿉니다.

### 1-1. 라이브러리와 데이터

준비 셀은 지난 시간과 같습니다(계산 장치와 그래프 글꼴).

데이터는 손글씨 숫자 [MNIST](https://web.archive.org/web/20231231235959/http://yann.lecun.com/exdb/mnist/)(LeCun, Cortes & Burges, CC BY-SA 3.0)에서 고른 학습용 10,000장과 테스트용 2,000장입니다. 지난 시간과 같은 표본입니다.

In [ ]:
import platform
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import torch

material_dir = Path(".")  # data가 있는 단원 폴더. 정답 노트북에서는 ".."
data_dir = material_dir / "data"

# 계산 장치: RunPod처럼 GPU(cuda)가 있으면 GPU, 없으면 CPU에서 계산합니다
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("계산 장치:", device, "|", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("계산 장치:", device)

# 그래프 글꼴: 영문, 숫자, 기호는 DejaVu Sans, 한글은 운영체제의 한글 글꼴이 맡습니다
from matplotlib import font_manager

if platform.system() == "Windows":
    korean_font = "Malgun Gothic"
elif platform.system() == "Darwin":
    korean_font = "AppleGothic"
else:
    korean_font = "NanumGothic"
    # 리눅스(RunPod): 글꼴을 설치하기 전에 만든 matplotlib 글꼴 목록에는 나눔고딕이 없으므로, 설치된 파일을 직접 등록합니다
    for font_path in font_manager.findSystemFonts():
        if "NanumGothic" in font_path:
            font_manager.fontManager.addfont(font_path)
plt.rcParams["font.family"] = ["DejaVu Sans", korean_font]
plt.rcParams["axes.unicode_minus"] = False  # 음수 부호가 네모로 깨지지 않게 합니다

In [ ]:
# torch.load는 그림과 정답 텐서 4개, 종류 이름 목록(class_names)이 든 딕셔너리를 돌려줍니다
mnist = torch.load(data_dir / "본실습" / "mnist_small.pt")
for name in ["train_images", "train_labels", "test_images", "test_labels"]:
    print(name, mnist[name].shape, mnist[name].dtype)
print("class_names:", mnist["class_names"])

In [ ]:
# 학습용 앞 10장을 정답과 함께 그립니다
class_names = mnist["class_names"]
plt.figure(figsize=(14, 2.2))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(mnist["train_images"][i], cmap="gray_r")  # gray_r: 0은 흰색, 255는 검은색
    plt.title(f"정답 {class_names[mnist['train_labels'][i]]}", fontsize=12)
    plt.axis("off")
plt.show()

### 1-2. CNN 입력 모양으로 바꾸기

`nn.Conv2d`는 그림 묶음을 4차원 텐서 (그림 수, 채널, 높이, 너비)로 받으므로, 28 × 28 그림 앞에 채널 축을 하나 끼웁니다.

- **채널(channel):** 한 자리(높이·너비의 한 위치)에 담긴 값의 개수입니다. 흑백 그림은 밝기 하나라 1, 컬러 사진은 빨강·초록·파랑 세 값이라 3이고, 합성곱 층의 출력에서는 특징 맵 수가 채널 수입니다(2-5).
- **`unsqueeze(1)`:** 1번 자리(그림 수 다음)에 크기 1인 축을 끼웁니다. 값은 그대로이고 모양만 바뀝니다.


<img src="images/61_image_tensor_shape.png" width="1050" alt="제목은 'CNN이 받는 그림 묶음의 모양: (그림 수, 채널, 높이, 너비)'입니다. 왼쪽에 손글씨 7이 그려진 28 x 28 흑백 그림 한 장이 있고 '흑백: 채널 1'이 붙어 있습니다. 그 옆에 빨강, 초록, 파랑 판 세 장이 겹친 컬러 사진이 있고 '컬러: 채널 3'이 붙어 있습니다. 오른쪽에는 흑백 그림 카드가 비스듬히 여러 장 쌓여 있고, 쌓인 방향에 '그림 수 10,000', 카드의 세로에 '높이 28', 가로에 '너비 28'이 적혀 있습니다. 아래 띠에 '(10,000, 28, 28) → unsqueeze(1) → (10,000, 1, 28, 28)'이 적혀 있습니다.">


1. **불러온 그림:** (10,000, 28, 28), 0~255 정수(`uint8`)입니다.
2. **255로 나누기:** 지난 시간처럼 0~1 실수(`float32`)로 바꿉니다. 0은 0.0, 255는 1.0입니다.
3. **채널 축 끼우기:** (10,000, 1, 28, 28)이 됩니다. 컬러 사진이었다면 (10,000, 3, 28, 28)입니다.

In [ ]:
# 0~1 실수로 바꾸고, 1번 자리에 채널 축(흑백이라 1)을 끼웁니다
X_train = (mnist["train_images"].float() / 255).unsqueeze(1)
X_test = (mnist["test_images"].float() / 255).unsqueeze(1)
y_train = mnist["train_labels"]
y_test = mnist["test_labels"]

# 바뀐 모양과 자료형, 픽셀 값 범위를 확인합니다
print("X_train:", X_train.shape, X_train.dtype, "| y_train:", y_train.shape, y_train.dtype)
print("X_test:", X_test.shape, "| y_test:", y_test.shape)
print("픽셀 최솟값:", X_train.min().item(), "| 최댓값:", X_train.max().item())

## 2. 합성곱

CNN의 첫 계산은 필터로 그림을 훑는 합성곱이므로, 5 × 5 작은 그림으로 계산을 한 칸씩 따라갑니다.

### 2-1. 필터 하나의 계산

- **합성곱(convolution):** 필터를 입력의 같은 크기 구역에 겹쳐, 같은 자리 값끼리 곱해 모두 더한 값 하나를 냅니다. 필터를 옮기며 모든 구역에서 이 계산을 합니다. 수학의 합성곱과 달리 필터를 뒤집지 않지만(교차 상관), 딥러닝과 `nn.Conv2d`는 이 계산을 합성곱이라 부릅니다.
- **필터(filter):** 가중치를 작은 사각형으로 묶은 것입니다(입력 채널마다 한 장씩). 크기는 `kernel_size`로 정하고(3 × 3, 5 × 5 등), 이 교안은 3 × 3입니다. 커널(kernel)이라고도 부릅니다. CNN은 이 가중치를 학습으로 정합니다.
- **특징 맵(feature map):** 필터 하나가 입력 전체를 훑어 만든 결과입니다. 필터가 찾는 모양이 있는 자리에서 값이 큽니다.
- **stride:** 필터를 한 번에 옮기는 칸 수입니다. 이 절의 합성곱은 모두 한 칸씩 옮깁니다(stride 1).

아래 움직이는 그림은 5 × 5 세로줄 그림에서 3 × 3 세로선 필터를 한 칸씩 옮기며, 자리마다 곱해서 더한 값으로 특징 맵 9칸을 하나씩 채우는 순서입니다.


<img src="images/53_convolution_steps_animation.gif" width="1050" alt="움직이는 그림입니다. 제목은 '합성곱: 필터를 옮기며 곱해서 더하기'입니다. 왼쪽 '입력 (5 x 5)'은 가운데 세로줄(2열)만 1이고 나머지는 0입니다. 가운데 '필터 (3 x 3)'는 세 행 모두 '-1, 2, -1'입니다. 빨간 3 x 3 테두리가 입력의 왼쪽 위에서 시작해 한 칸씩 오른쪽, 다음 줄로 옮겨 가고(stride 1, 위치 1 / 9부터 9 / 9), 테두리 안 칸마다 곱할 필터 값(×(-1), ×2)이 작게 적힙니다. 아래 계산 띠에 '행마다: 0×(-1) + 0×2 + 1×(-1) = -1, 세 행 더하기: (-1) + (-1) + (-1) = -3'처럼 계산이 나오고, 오른쪽 '특징 맵 (3 x 3)'의 해당 칸에 값이 들어갑니다. 마지막 화면의 특징 맵은 세 행 모두 '-3, 6, -3'이고, 띠에 '가운데 열 6: 세로줄이 필터 가운데(2)에 겹친 자리', '양옆 열 -3: 세로줄이 필터 가장자리(-1)에 걸린 자리'가 적혀 있습니다.">

<img src="images/62_convolution_steps.png" width="1050" alt="제목은 '합성곱: 필터를 겹쳐 곱해서 더하고, 옮겨 가며 반복합니다'입니다. 왼쪽에 5 x 5 그림이 있고 가운데 세로줄(2열) 다섯 칸만 1, 나머지는 0입니다. 그림 왼쪽 위 3 x 3 구역에 주황 테두리 1, 한 칸 오른쪽 구역에 초록 테두리 2가 겹쳐 있습니다. 가운데에 3 x 3 필터 '-1, 2, -1 / -1, 2, -1 / -1, 2, -1'이 있습니다. 오른쪽 3 x 3 특징 맵은 세 행 모두 '-3, 6, -3'이고, 첫 칸 -3은 주황, 둘째 칸 6은 초록으로 칠해져 있습니다. 아래 계산 띠 두 줄: '주황: 행마다 0 × (-1) + 0 × 2 + 1 × (-1) = -1, 세 행 더해 -3', '초록: 행마다 0 × (-1) + 1 × 2 + 0 × (-1) = 2, 세 행 더해 6'.">


가운데 세로줄만 칠해진 5 × 5 그림(칠해진 칸 1, 빈칸 0)에 세로선 필터를 씁니다. 필터의 가운데 열은 2, 양옆 열은 -1입니다.

1. **왼쪽 위 구역(0~2행, 0~2열):** 행마다 픽셀 [0, 0, 1]과 필터 [-1, 2, -1]을 곱해 더하면 -1이고, 세 행을 더하면 -3입니다.
2. **한 칸 오른쪽 구역(0~2행, 1~3열):** 행마다 [0, 1, 0]이라 2이고, 세 행을 더하면 6입니다. 세로줄이 필터 가운데 열에 딱 겹친 자리입니다.
3. **결과 크기:** 5 × 5 그림에서 3 × 3 필터가 놓일 자리는 가로 3곳, 세로 3곳이라 특징 맵은 3 × 3입니다.

In [ ]:
# 5 x 5 그림: 가운데 세로줄(2열)만 1입니다
image_v = torch.zeros(5, 5)
image_v[:, 2] = 1
display(image_v)

# 세로선 필터: 가운데 열은 2, 양옆 열은 -1
filter_v = torch.tensor([[-1.0, 2.0, -1.0],
                         [-1.0, 2.0, -1.0],
                         [-1.0, 2.0, -1.0]])
display(filter_v)

In [ ]:
def convolve(image, kernel):
    """3 x 3 필터를 한 칸씩 옮기며 곱해서 더한 특징 맵을 돌려줍니다."""
    # 필터가 놓일 자리 수: 5 x 5 그림이면 5 - 3 + 1 = 3
    size = image.shape[0] - 2
    feature_map = torch.zeros(size, size)
    for row in range(size):
        for col in range(size):
            patch = image[row:row + 3, col:col + 3]  # 필터와 겹치는 3 x 3 구역
            feature_map[row, col] = (patch * kernel).sum()  # 같은 자리끼리 곱해 모두 더합니다
    return feature_map


# 세로줄 그림에 세로선 필터를 씁니다
v_result = convolve(image_v, filter_v)
display(v_result)

**결과 읽기**
- **가운데 열 6:** 세로줄이 필터 가운데(2)에 겹친 자리라 값이 가장 큽니다.
- **양옆 열 -3:** 세로줄이 필터 가장자리(-1)에 걸린 자리라 음수입니다. 세 행이 모두 [-3, 6, -3]으로 같습니다.

### 2-2. 세로선 필터와 가로선 필터

필터의 가중치 배치가 찾는 모양을 정합니다. 세로선 필터를 눕힌 가로선 필터를 만들어 세로줄 그림과 가로줄 그림에 각각 쓰면 결과는 아래 표와 같습니다.

- **가로선 필터:** 가운데 행은 2, 위아래 행은 -1입니다.
- **세로선 필터에 가로줄이 겹치면:** 한 행의 세 칸이 모두 1이라 -1 + 2 - 1 = 0입니다. 필터 방향과 다른 선에는 반응하지 않습니다.

| 그림 | 세로선 필터 | 가로선 필터 |
|---|---|---|
| 세로줄 그림 | 가운데 열 6 | 모두 0 |
| 가로줄 그림 | 모두 0 | 가운데 행 6 |

같은 두 필터를 진짜 손글씨 7(학습용 1번 그림)에 씁니다. 그림이 28 × 28이라 필터가 놓일 자리는 가로·세로 28 - 3 + 1 = 26곳씩, 특징 맵은 26 × 26입니다. 음수는 지난 시간의 ReLU로 0으로 바꿔 필터가 찾은 자리만 남깁니다.

In [ ]:
# 가로선 필터: 세로선 필터를 눕힌 모양(가운데 행은 2, 위아래 행은 -1)
filter_h = torch.tensor([[-1.0, -1.0, -1.0],
                         [2.0, 2.0, 2.0],
                         [-1.0, -1.0, -1.0]])

# 학습용 1번 그림(숫자 7)의 28 x 28 픽셀에 두 필터를 쓰고, ReLU로 음수를 0으로 바꿉니다
digit = X_train[1, 0]
v_digit_map = torch.relu(convolve(digit, filter_v))
h_digit_map = torch.relu(convolve(digit, filter_h))
print("원본:", tuple(digit.shape), "| 특징 맵:", tuple(v_digit_map.shape))

# 왼쪽부터 원본, 세로선 필터의 특징 맵, 가로선 필터의 특징 맵
plt.figure(figsize=(14, 4.8))
panels = {"원본 (정답 7)": digit, "세로선 필터의 특징 맵": v_digit_map, "가로선 필터의 특징 맵": h_digit_map}
for k, title in enumerate(panels):
    plt.subplot(1, 3, k + 1)
    plt.imshow(panels[title], cmap="gray_r")
    plt.title(title, fontsize=14)
    plt.axis("off")
plt.show()

**그래프 읽기**
- **세로선 필터:** 오른쪽 위에서 왼쪽 아래로 내려오는 긴 획이 진하게 남고, 맨 위 가로획은 양 끝만 조금 남습니다.
- **가로선 필터:** 맨 위 가로획이 진하게 남고, 긴 세로 방향 획은 흐려집니다.
- 같은 그림이라도 필터마다 다른 획을 골라냅니다. CNN은 이런 필터의 가중치를 사람이 정하지 않고 학습으로 정합니다.

### 2-3. nn.Conv2d로 같은 계산 확인

PyTorch의 합성곱 층이 위에서 직접 한 계산과 같은지, 같은 가중치를 넣어 확인합니다.

- **`nn.Conv2d(in_channels, out_channels, kernel_size)`:** 합성곱 층입니다. 입력 채널 수, 출력 채널 수(필터 수), 필터 크기를 받습니다. 필터의 가중치(`weight`)와 절편(`bias`)이 파라미터입니다.
- **`weight` 모양:** (출력 채널, 입력 채널, 필터 높이, 필터 너비)입니다. 3 × 3 필터 1개가 흑백 1채널을 보면 (1, 1, 3, 3)입니다.
- 학습 대신 세로선 필터의 가중치를 `weight`에 직접 넣고, 필터 계산만 비교하려고 절편은 뺍니다(`bias=False`).

In [ ]:
import torch.nn as nn

# 입력 채널 1개, 출력 채널 1개(필터 1개), 3 x 3 필터인 합성곱 층
conv = nn.Conv2d(1, 1, kernel_size=3, bias=False)
print("weight 모양:", conv.weight.shape)

# 세로선 필터의 가중치를 (출력 채널, 입력 채널, 3, 3) 모양으로 넣습니다
with torch.no_grad():
    conv.weight[:] = filter_v.reshape(1, 1, 3, 3)

    # 5 x 5 그림도 (그림 수 1, 채널 1, 5, 5) 모양으로 넣습니다
    conv_output = conv(image_v.reshape(1, 1, 5, 5))
display(conv_output)
print("직접 계산과 같은가:", torch.equal(conv_output[0, 0], v_result))

**결과 읽기**
- **직접 계산과 같은가 True:** 손으로 짠 `convolve`와 `nn.Conv2d`가 같은 값을 냅니다. 합성곱 층은 위에서 계산한 일을 그대로 합니다.

### 2-4. padding과 출력 크기

필터가 놓일 자리만 계산하면 층을 지날 때마다 가장자리가 깎이므로(3 × 3 필터는 둘레 한 줄씩), 그림 둘레에 0을 둘러 크기를 지킵니다.

- **padding:** 입력 테두리에 두르는 칸의 겹 수입니다. `nn.Conv2d`는 0으로 채우고(`padding_mode="zeros"`), 기본은 `padding=0`이라 두르지 않습니다. `padding=1`이면 한 겹이라 5 × 5 그림이 7 × 7이 되고, 가장자리 픽셀도 필터 가운데에 놓입니다.
- **출력 크기:** 입력 크기 $n$, 필터 크기 $f$, padding $p$일 때 출력 크기 $o$는, 필터가 놓일 자리 수 $n - f + 1$에 양쪽 padding $2p$칸을 더한 값입니다(stride 1).

$\displaystyle o = n - f + 1 + 2p$

아래 움직이는 그림은 2-1의 세로줄 그림에 `padding=1`을 둘러 7 × 7로 만든 뒤, 세로선 필터를 한 칸씩 옮기며 5 × 5 특징 맵을 채우는 순서입니다.


<img src="images/57_padding_animation.gif" width="1050" alt="움직이는 그림입니다. 제목은 'padding=1: 테두리에 0을 둘러 5 x 5 입력이 5 x 5 특징 맵이 됩니다'입니다. 왼쪽 '입력 5 x 5 + padding 1 = 7 x 7'은 회색 테두리 칸이 모두 0이고, 안쪽 5 x 5는 가운데 열만 1입니다. 가운데에 '필터 3 x 3' 세로선 필터 '-1, 2, -1'(세 행 같음)이 있습니다. 빨간 3 x 3 테두리가 7 x 7의 왼쪽 위에서 시작해 한 칸씩 오른쪽, 다음 줄로 옮겨 가고, 오른쪽 '특징 맵 5 x 5'가 한 칸씩 채워집니다. 맨 위 줄과 맨 아래 줄은 '0, -2, 4, -2, 0', 가운데 세 줄은 '0, -3, 6, -3, 0'입니다. 아래 띠에 지금 칸의 값과 '필터 일부가 padding의 0에 걸림' 또는 '필터가 원래 그림 안에 놓임'이 나옵니다. 마지막 장에서 입력의 안쪽 5 x 5와 특징 맵의 가운데 3 x 3에 초록 테두리가 생기고, 띠에 'padding=1: 5 - 3 + 1 + 2 × 1 = 5 | padding=0: 5 - 3 + 1 = 3', 'padding=0이면 초록 테두리 안의 가운데 3 x 3만 나옵니다'가 적힙니다.">

<img src="images/63_padding.png" width="1050" alt="제목은 'padding: 테두리에 0을 둘러 출력 크기를 조절합니다'입니다. 왼쪽 판 'padding=0'에는 '5 x 5' 그림의 왼쪽 위에 3 x 3 필터가 겹쳐 있고, 결과 '3 x 3' 특징 맵과 '5 - 3 + 1 = 3'이 있습니다. 오른쪽 판 'padding=1'에는 5 x 5 둘레에 0 칸을 한 겹 둘러 '7 x 7'이 된 그림이 있습니다. 필터는 맨 왼쪽 위 모서리의 0 칸에 걸쳐 있고, 필터 가운데 칸인 원래 그림의 왼쪽 위 픽셀에 '가장자리 픽셀이 필터 가운데' 말풍선이 붙어 있습니다. 결과 '5 x 5' 특징 맵과 '5 - 3 + 1 + 2 × 1 = 5'가 이어지고, 아래 띠에 '출력 크기 = 입력 - 필터 + 1 + 2 × padding (stride 1)'이 적혀 있습니다.">


1. **5 × 5, padding 0:** 5 - 3 + 1 = 3입니다. 앞에서 직접 계산한 특징 맵이 3 × 3이었던 까닭입니다.
2. **5 × 5, padding 1:** 5 - 3 + 1 + 2 = 5로 크기가 그대로입니다.
3. **28 × 28, padding 1:** 28 그대로입니다. 이 교안의 CNN은 합성곱에서 `padding=1`로 크기를 지키고, 크기를 줄이는 일은 다음 절의 풀링에 맡깁니다.

### 2-5. 채널과 특징 맵

필터 하나는 한 가지 모양만 찾으므로, 실제 CNN은 한 층에 필터를 여러 개 두어 여러 특징을 함께 찾습니다.

- **출력 채널 수 = 필터 수 = 특징 맵 수:** `nn.Conv2d(1, 16, ...)`은 필터 16개가 각자 그림을 훑어 특징 맵 16장을 만듭니다. 그림 64장이면 (64, 16, 28, 28)입니다.
- **다음 층의 필터:** 필터는 입력 채널을 모두 덮으므로, `nn.Conv2d(16, 32, kernel_size=3)`의 필터 하나는 가중치 3 × 3 × 16개입니다. 특징 맵 16장의 같은 3 × 3 자리를 한꺼번에 곱해 모두 더하므로, 앞 층이 찾은 여러 특징을 묶어 더 복잡한 모양을 찾을 수 있습니다.

아래 움직이는 그림은 입력 채널 2장(4 × 4)에 3 × 3 필터 2개를 쓰는 작은 예로, 필터 하나가 채널마다 조각 하나씩을 곱해 더하고 절편까지 더해 특징 맵 한 칸을 채우는 순서입니다.


<img src="images/58_channels_animation.gif" width="1050" alt="움직이는 그림입니다. 제목은 '필터 하나가 입력 채널을 모두 덮고, 채널별 결과를 더해 값 하나를 냅니다'입니다. 왼쪽에 '입력 채널 1' 4 x 4 '1, 0, 2, 1 / 0, 1, 1, 0 / 2, 1, 0, 1 / 1, 0, 1, 2'와 '입력 채널 2' 4 x 4 '0, 2, 1, 0 / 1, 0, 0, 1 / 0, 1, 2, 1 / 2, 1, 0, 0'이 위아래로 있습니다. 그 오른쪽에 지금 필터의 '조각 1'과 '조각 2'(3 x 3)가 채널 옆에 하나씩 있고, 맨 오른쪽에 '특징 맵 1'과 '특징 맵 2'(2 x 2)가 있습니다. 필터 1(조각 1 '1, 0, -1' 세 행, 조각 2 '0, 1, 0 / 1, -1, 1 / 0, 1, 0', 절편 +1)을 소개하는 장 뒤로, 두 입력 채널의 같은 3 x 3 자리에 빨간 테두리가 함께 놓여 한 칸씩 옮겨 갑니다. 장마다 '채널 1: 0', '채널 2: 4'처럼 채널별로 곱해 더한 값과 '0 + 4 + 1 = 5' 같은 합이 나오고, 특징 맵 1이 '5, 5 / 4, 0'으로 채워집니다. 이어서 필터 2(조각 1 '0, 0, 0 / 1, 1, 1 / 0, 0, 0', 조각 2 '-1, 0, 1 / 0, 1, 0 / 1, 0, -1', 절편 -1)가 같은 방식으로 특징 맵 2를 '0, -1 / 4, 5'로 채웁니다. 마지막 장 띠에 'nn.Conv2d(2, 2, kernel_size=3): weight 모양 (2, 2, 3, 3)', '필터 2개 -> 특징 맵 2장, 출력 모양 (2, 2, 2)'가 적혀 있습니다.">

<img src="images/64_channels.png" width="1050" alt="제목은 '채널: 필터 수만큼 특징 맵이 생깁니다'입니다. 왼쪽에 손글씨 7 흑백 그림 한 장 '(1, 28, 28)'이 있습니다. 그 오른쪽에 색이 다른 작은 3 x 3 필터들이 세로로 늘어서 있고 'nn.Conv2d(1, 16): 필터 16개'가 적혀 있으며, 필터마다 화살표가 한 묶음으로 쌓인 특징 맵 '(16, 28, 28)'로 모입니다. 다음에 3 x 3 칸이 층층이 쌓인 기둥 모양 필터 '3 x 3 x 16'과 'nn.Conv2d(16, 32): 필터 하나가 16장을 한꺼번에 봅니다'가 있습니다. 맨 오른쪽에 더 두껍게 쌓인 특징 맵 묶음 '(32, 28, 28)'과 '필터 32개 -> 특징 맵 32장'이 있습니다.">


1. **첫째 층 `nn.Conv2d(1, 16)`:** 필터 16개 × 가중치 9개(1 × 3 × 3) = 가중치 144개, 절편 16개를 더해 파라미터 160개입니다.
2. **둘째 층 `nn.Conv2d(16, 32)`:** 필터 32개 × 가중치 144개(16 × 3 × 3) = 가중치 4,608개, 절편 32개를 더해 4,640개입니다.
3. **모양:** 그림 64장이 (64, 1, 28, 28) → (64, 16, 28, 28) → (64, 32, 28, 28)로 바뀝니다. 그림 수와 크기는 그대로이고 채널 수만 바뀝니다.

### 2-6. 지역 연결과 파라미터 공유

둘째 층은 28 × 28 그림 16장을 받는데도 파라미터가 4,640개뿐입니다. 합성곱 층의 두 성질 때문이고, 같은 입출력을 `nn.Linear`로 이었을 때와 비교하면 차이가 드러납니다.

- **지역 연결(local connectivity):** 출력 한 칸이 입력 전체가 아니라 같은 자리 둘레의 작은 구역(이 층은 3 × 3, 채널 16장)만 보고 계산됩니다. `nn.Linear`는 출력 한 칸이 입력의 모든 값과 연결됩니다.
- **파라미터 공유(parameter sharing):** 같은 필터를 그림 전체에 밀며 모든 자리에서 같은 가중치를 씁니다. 그래서 가중치 수가 그림 크기와 상관없이 필터 수, 필터 크기, 입력 채널 수로 정해집니다.
- **목적:** 가중치 수를 크게 줄이고, 같은 모양이 그림 어디에 있어도 같은 필터로 찾게 합니다. 숫자가 몇 칸 옮겨지면 특징 맵의 큰 값도 같은 만큼 옮겨질 뿐, 같은 필터가 같은 특징을 찾습니다.


<img src="images/52_local_shared_weights.png" width="1050" alt="제목은 '지역 연결과 파라미터 공유: 가중치 수 비교'이고, 제목 아래 띠에 '입력 (16, 28, 28) -> 출력 (32, 28, 28)'이 적혀 있습니다. 카드 세 장이 왼쪽에서 오른쪽으로 화살표로 이어지고, 카드마다 왼쪽에 '입력', 오른쪽에 '출력' 칸 묶음이 있습니다. 1 '모두 연결 (Linear)' 카드에서는 출력 한 칸에서 입력의 모든 칸으로 선이 뻗고, 칩 '출력 한 칸이 입력 전체를 봄'과 상자 '가중치 314,703,872개'가 있습니다. 2 '지역 연결' 카드에서는 빨간 출력 칸이 입력의 왼쪽 위 3 x 3 구역에만, 초록 출력 칸이 오른쪽 아래 3 x 3 구역에만 이어지고 두 구역의 필터 색이 다르며, 칩 '자리마다 다른 가중치'와 상자 '가중치 3,612,672개'가 있습니다. 3 '파라미터 공유' 카드에서는 '같은 필터' 하나가 두 구역에 모두 쓰이고, 칩 '모든 자리에서 같은 가중치'와 상자 '가중치 4,608개'가 있습니다. 맨 아래 띠에 '같은 모양이 그림 어디에 있어도 같은 필터로 찾음'이 적혀 있습니다.">


2-5의 둘째 층과 같은 입출력 (16, 28, 28) → (32, 28, 28)에서 가중치 수(절편 제외)를 세 단계로 줄여 봅니다.

1. **모두 연결(`nn.Linear`):** 입력 16 × 28 × 28 = 12,544개와 출력 32 × 28 × 28 = 25,088개를 모두 이으면 12,544 × 25,088 = 314,703,872개(약 3억 1,470만 개)입니다.
2. **지역 연결만:** 출력 한 칸이 입력 3 × 3 × 16 = 144개만 보면, 출력 칸마다 따로 가중치를 두어도 25,088 × 144 = 3,612,672개입니다.
3. **파라미터 공유까지:** 필터 32개를 28 × 28 = 784자리에서 함께 쓰므로 32 × 144 = 4,608개이고, 절편 32개를 더하면 2-5에서 계산한 4,640개입니다. 모두 연결한 경우의 약 6만 8천분의 1입니다.

## 3. 풀링

특징 맵이 32장으로 늘면 계산할 값도 그만큼 늘어나므로, 찾은 특징은 남기고 크기를 줄입니다.

### 3-1. 2 × 2 최댓값 풀링

- **풀링(pooling):** 특징 맵의 작은 구역마다 값 하나(최댓값, 평균 등)만 남겨 크기를 줄이는 계산입니다. 채널마다 따로 합니다. 파라미터가 없습니다.
- **최댓값 풀링(max pooling):** 구역마다 가장 큰 값을 남깁니다. 이 교안의 `nn.MaxPool2d(2)`는 2 × 2 구역을 2칸씩 옮기므로(stride 2, 따로 주지 않으면 구역 크기와 같음) 구역이 겹치지 않고, 가로와 세로가 절반, 값의 개수는 4분의 1이 됩니다.
- **최댓값을 남기는 이유:** 특징 맵의 큰 값은 필터가 찾는 모양이 있다는 뜻입니다. 그 모양이 같은 구역 안에서 옮겨지면 최댓값이 그대로라, 획의 위치가 조금 달라져도 출력이 덜 바뀝니다.

아래 움직이는 그림은 4 × 4 특징 맵을 2 × 2 구역으로 나눠 2칸씩 옮기며, 구역마다 가장 큰 값을 골라 결과 2 × 2를 채우는 순서입니다.


<img src="images/54_max_pooling_steps_animation.gif" width="1050" alt="움직이는 그림입니다. 제목은 '최댓값 풀링: 구역마다 가장 큰 값만 남기기'입니다. 왼쪽 '특징 맵 (4 x 4)' '1, 3, 0, 2 / 4, 2, 1, 0 / 0, 1, 5, 6 / 2, 0, 7, 1'이 2 x 2 구역 네 개(파랑, 주황, 초록, 보라)로 나뉘어 있고, 위에 '2 x 2 구역, 2칸씩 옮김 (stride 2)'가 적혀 있습니다. 빨간 테두리가 왼쪽 위, 오른쪽 위, 왼쪽 아래, 오른쪽 아래 구역 순서로 옮겨 가며 가장 큰 값에 빨간 동그라미를 치고, 아래 띠에 'max(1, 3, 4, 2) = 4'처럼 계산이 나온 뒤 빨간 화살표를 따라 오른쪽 '결과 (2 x 2)' 칸에 같은 색으로 들어갑니다. 마지막 화면의 결과는 '4, 2 / 2, 7'이고, 띠에 '가로, 세로가 절반: 4 x 4 → 2 x 2', '값 16개 → 4개 (4분의 1)'가 적혀 있습니다.">

<img src="images/65_max_pooling.png" width="1050" alt="제목은 '최댓값 풀링: 구역마다 가장 큰 값만 남깁니다'입니다. 왼쪽에 4 x 4 특징 맵 '1, 3, 0, 2 / 4, 2, 1, 0 / 0, 1, 5, 6 / 2, 0, 7, 1'이 있고, 2 x 2 구역 네 개가 각각 파랑, 주황, 초록, 보라 배경으로 칠해져 있습니다. 구역마다 가장 큰 값 4, 2, 2, 7에 동그라미가 쳐져 있습니다. 오른쪽에 같은 네 색의 2 x 2 결과 '4, 2 / 2, 7'이 있고, 가운데 화살표 위에 'nn.MaxPool2d(2)'가 적혀 있습니다. 아래 띠에 '가로, 세로가 절반: 28 x 28 → 14 x 14 → 7 x 7'이 적혀 있습니다.">


1. **왼쪽 위 구역** [1, 3, 4, 2]의 최댓값은 4, **오른쪽 위** [0, 2, 1, 0]은 2입니다.
2. **왼쪽 아래** [0, 1, 2, 0]은 2, **오른쪽 아래** [5, 6, 7, 1]은 7입니다.
3. 4 × 4 특징 맵이 2 × 2 [[4, 2], [2, 7]]이 됩니다.

In [ ]:
# 4 x 4 특징 맵 예시
feature = torch.tensor([[1.0, 3.0, 0.0, 2.0],
                        [4.0, 2.0, 1.0, 0.0],
                        [0.0, 1.0, 5.0, 6.0],
                        [2.0, 0.0, 7.0, 1.0]])

# nn.MaxPool2d(2): 2 x 2 구역의 최댓값 풀링 층. 입력은 (그림 수, 채널, 높이, 너비) 모양입니다
pool = nn.MaxPool2d(2)
pool_output = pool(feature.reshape(1, 1, 4, 4))
display(pool_output)

### 3-2. 손글씨 특징 맵에 풀링하기

28 × 28 특징 맵에 풀링을 두 번 하면 14 × 14, 7 × 7이 됩니다. 숫자 7의 세로선 특징 맵(`padding=1`이라 28 × 28)으로 크기가 줄어도 획 모양이 남는지 봅니다.

In [ ]:
# 세로선 필터를 넣은 합성곱 층(padding=1)으로 숫자 7의 28 x 28 특징 맵을 만들고 ReLU를 씁니다
v_conv = nn.Conv2d(1, 1, kernel_size=3, padding=1, bias=False)
with torch.no_grad():
    v_conv.weight[:] = filter_v.reshape(1, 1, 3, 3)
    v_map = torch.relu(v_conv(X_train[1:2]))  # X_train[1:2]: 숫자 7 한 장, 모양 (1, 1, 28, 28)

    # 최댓값 풀링을 두 번 합니다
    pooled_once = pool(v_map)
    pooled_twice = pool(pooled_once)

# 왼쪽부터 풀링 전 28 x 28, 한 번 뒤 14 x 14, 두 번 뒤 7 x 7
plt.figure(figsize=(14, 4.8))
panels = {"풀링 전": v_map, "풀링 1번 뒤": pooled_once, "풀링 2번 뒤": pooled_twice}
for k, title in enumerate(panels):
    plt.subplot(1, 3, k + 1)
    plt.imshow(panels[title][0, 0], cmap="gray_r")
    plt.title(f"{title}: {tuple(panels[title].shape[2:])}", fontsize=14)
    plt.axis("off")
plt.show()

**그래프 읽기**
- **칸 수:** 784개에서 196개, 49개로 줄어듭니다.
- **획의 자리:** 7 × 7에서도 왼쪽 아래로 내려오는 긴 획의 자리가 진한 칸으로 남습니다. 뒤 층은 이 작은 특징 맵으로 어디쯤에 어떤 획이 있는지 계산합니다.

## 4. CNN 쌓기

합성곱으로 특징을 찾고 풀링으로 줄이는 묶음을 두 번 쌓은 뒤, 마지막에 지난 시간처럼 `nn.Linear`로 숫자 0~9의 로짓 10개를 냅니다.

### 4-1. nn.Sequential로 CNN 만들기

- **합성곱 묶음:** `nn.Conv2d` → `nn.ReLU` → `nn.MaxPool2d(2)`입니다. 합성곱도 곱해서 더하는 선형 계산이라, 지난 시간처럼 층 사이에 ReLU를 넣습니다.
- **`nn.Flatten()`:** 마지막 특징 맵 (32, 7, 7)을 숫자 1,568개(32 × 7 × 7) 한 줄로 펼칩니다. 합성곱 층이 이웃 픽셀로 특징을 찾은 뒤에 펼치므로, 처음부터 픽셀을 펼친 DNN과 다릅니다.
- **로짓(logit):** 출력층이 종류마다 내는 softmax 전의 점수입니다. softmax를 거치면 종류별 확률이 됩니다.
- **손실:** 지난 시간 다중 분류의 짝(로짓 10개 + `nn.CrossEntropyLoss`)을 그대로 씁니다.


<img src="images/66_cnn_structure.png" width="1050" alt="제목은 'CNN: 특징을 찾고 줄이기를 두 번, 마지막에 분류'입니다. 위쪽 '특징 찾기' 판에서 손글씨 7 그림 '(1, 28, 28)'이 파란 상자 '합성곱 묶음 1: Conv2d(1, 16) + ReLU + MaxPool2d(2)'를 지나 판 16장 '(16, 14, 14)'가 되고, '합성곱 묶음 2: Conv2d(16, 32) + ReLU + MaxPool2d(2)'를 지나 판 32장 '(32, 7, 7)'이 됩니다. 화살표가 아래쪽 '분류' 판으로 이어져 회색 상자 'Flatten: 1,568개 한 줄', 초록 상자 'Linear(1568, 10)'을 지나 '로짓 10개' 막대그래프(0~9, 7 막대만 주황으로 가장 김)가 됩니다.">

In [ ]:
def make_cnn():
    """합성곱 묶음 두 개(필터 16개, 32개)와 출력층으로 된 CNN을 만들어 계산 장치에 올려 돌려줍니다."""
    return nn.Sequential(
        nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # (16, 14, 14)
        nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # (32, 7, 7)
        nn.Flatten(),                                                             # 1,568개 한 줄
        nn.Linear(32 * 7 * 7, 10),                                                # 로짓 10개
    ).to(device)


# 파라미터 초깃값을 고정해 다시 실행해도 같은 모델이 나오게 합니다
torch.manual_seed(0)
model = make_cnn()
print(model)

In [ ]:
# 층마다 파라미터 모양과 개수. 이름의 앞 숫자는 nn.Sequential 안의 순서입니다
param_rows = []
for name, param in model.named_parameters():
    param_rows.append({"이름": name, "모양": tuple(param.shape), "개수": param.numel()})
param_table = pd.DataFrame(param_rows)
display(param_table)
print("파라미터 수 합계:", param_table["개수"].sum())

**결과 읽기**
- **합성곱 층(`0`, `3`):** 160개와 4,640개입니다. 파라미터 공유(2-6)로 필터의 가중치를 그림의 모든 자리에서 함께 쓰므로, 그림이 커져도 개수가 늘지 않습니다.
- **출력층(`7.weight`):** (10, 1568), 15,680개로 가장 많습니다.
- **합계 20,490개:** 지난 시간 DNN(109,386개)의 약 5분의 1입니다.

### 4-2. 층마다 모양 변화

어느 층에서 모양이 어떻게 바뀌는지 알아야 `nn.Linear`의 입력 크기(32 × 7 × 7)를 맞출 수 있으므로, 미니배치 하나를 층마다 차례로 통과시켜 모양을 기록합니다. 미니배치를 꺼내는 `DataLoader`는 지난 시간처럼 64장씩, 학습용만 섞어 만들고, 학습용에는 `drop_last=True`를 더합니다.

- **`drop_last=True`:** 그림 수가 배치 크기로 나누어떨어지지 않을 때 남는 마지막 작은 미니배치를 버립니다(학습용 10,000장을 64장씩이면 156개, 남는 16장은 에폭마다 섞인 순서의 끝에서 빠짐). 5절의 배치 정규화는 미니배치의 평균과 분산을 쓰므로 작은 미니배치가 이 값을 흔들고 채널마다 값이 하나뿐이면 학습 모드에서 오류가 나며, 테스트용은 모든 그림을 평가해야 하므로 버리지 않습니다.

아래 움직이는 그림은 20장을 6장씩 자르는 작은 예로, 에폭마다 섞은 순서를 미니배치로 자르고 끝에 남는 2장을 버리는 순서입니다.


<img src="images/59_minibatch_animation.gif" width="1050" alt="움직이는 그림입니다. 제목은 '미니배치: 에폭마다 섞어 6장씩 자르고, 남는 2장은 버립니다'입니다. 1부터 20까지 번호 카드 20장이 한 줄로 놓였다가 '에폭 1: 섞은 순서 (shuffle=True)'로 순서가 바뀌고, 빨간 테두리가 앞에서부터 6장씩 옮겨 가며 '미니배치 1', '미니배치 2', '미니배치 3'으로 묶습니다. 끝에 남은 4번, 3번 카드는 회색으로 바뀌고 '버림'이 붙습니다. 아래 줄 '에폭 2'에서는 다른 순서로 섞여 같은 방식으로 잘리고 7번, 12번 카드를 버립니다. 마지막 장에 '에폭마다 섞는 순서가 달라 버리는 그림도 다릅니다: 에폭 1은 4, 3번 / 에폭 2는 7, 12번'이 빨간 글자로 나옵니다. 아래 띠는 내내 '이 그림: 20장을 6장씩 -> 미니배치 3개, 남는 2장 버림', '이 교안: 학습용 10,000장을 64장씩 -> 미니배치 156개, 남는 16장 버림 (drop_last=True)'입니다.">

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

# 학습용은 에폭마다 섞고 마지막 작은 미니배치는 버립니다. 테스트용은 결과만 계산하므로 섞지 않고 모두 씁니다
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True, drop_last=True)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=64, shuffle=False)
print("학습 미니배치 수:", len(train_loader), "| 테스트 미니배치 수:", len(test_loader))

In [ ]:
# 학습용 미니배치 하나(64장)를 꺼냅니다
for X_batch, y_batch in train_loader:
    break

# nn.Sequential의 층을 하나씩 꺼내 차례로 통과시키며 출력 모양을 기록합니다
shape_rows = [{"번호": "-", "층": "입력", "출력 모양": tuple(X_batch.shape)}]
x = X_batch.to(device)
with torch.no_grad():
    for k, layer in enumerate(model):
        x = layer(x)
        shape_rows.append({"번호": k, "층": type(layer).__name__, "출력 모양": tuple(x.shape)})  # 층 종류 이름만 적습니다

# 층마다 출력 모양을 표로 봅니다
shape_table = pd.DataFrame(shape_rows)
display(shape_table)

**결과 읽기**
- **번호:** `print(model)`의 (0)~(7)과 같습니다.
- **합성곱(`Conv2d`, 0번과 3번):** 채널 수만 1 → 16, 16 → 32로 바뀌고 크기(28, 14)는 그대로입니다(`padding=1`).
- **풀링(`MaxPool2d`, 2번과 5번):** 채널 수는 그대로이고 크기만 28 → 14, 14 → 7로 절반이 됩니다(stride 2).
- **`Flatten`(6번)과 `Linear`(7번):** (64, 32, 7, 7)이 (64, 1568)로 펼쳐지고, 마지막에 그림 64장마다 로짓 10개인 (64, 10)이 됩니다. `nn.Linear`의 입력 크기를 1,568이 아닌 값으로 적으면 이 줄에서 모양이 맞지 않는다는 에러가 납니다.

### 4-3. 수용 영역

이 교안의 필터는 3 × 3뿐인데 뒤 층이 획을 여러 개 묶은 큰 모양을 찾을 수 있는 것은, 층을 지날 때마다 특징 맵 한 칸이 보는 원본 구역이 넓어지기 때문입니다.

- **수용 영역(receptive field):** 특징 맵 한 칸의 값을 계산하는 데 쓰인 원래 입력 그림의 구역입니다. 그 구역 밖의 픽셀은 이 칸의 값을 바꾸지 않습니다.


<img src="images/51a_receptive_field_layers.png" width="1050" alt="제목은 'Receptive Field in Convolutional Networks'입니다. 칸으로 나뉜 판 세 장이 비스듬히 왼쪽부터 'Layer 1', 'Layer 2', 'Layer 3' 순서로 놓여 있습니다. Layer 3 가운데쯤의 빨간 한 칸에서 점선 네 개가 Layer 2의 주황 3 x 3 구역 네 모서리로 이어집니다. 그 주황 구역의 왼쪽 위 한 칸은 빨간색이고, 이 빨간 칸에서 나온 점선은 Layer 1의 왼쪽 위 빨간 3 x 3 구역으로 이어집니다. Layer 2의 주황 3 x 3 구역에서 나온 점선은 Layer 1의 주황 5 x 5 구역으로 이어지고, 빨간 3 x 3 구역은 이 5 x 5 구역의 왼쪽 위 모서리에 들어 있습니다.">


- **2층의 한 칸:** Layer 2의 빨간 한 칸은 Layer 1의 3 × 3(빨강)을 봅니다.
- **3층의 한 칸:** Layer 3의 빨간 한 칸은 Layer 2의 3 × 3(주황)을 보고, 그 9칸이 각자 Layer 1의 3 × 3을 보므로 Layer 1에서는 5 × 5(주황)를 봅니다. 3 × 3 필터, stride 1 합성곱을 하나 쌓을 때마다 2칸씩 넓어집니다(3 → 5 → 7).
- **층이 깊어지면:** 한 칸이 보는 원본 구역이 넓어지므로, 앞 층은 짧은 획을, 뒤 층은 획을 묶은 큰 모양(숫자의 윗부분, 고리)을 찾습니다.

풀링(stride 2)이 끼면 더 빨리 넓어져, 이 교안 CNN의 마지막 특징 맵 한 칸은 원본 28 × 28의 10 × 10 구역을 봅니다(3 → 4 → 8 → 10).


<img src="images/51_receptive_field.png" width="1050" alt="제목은 '수용 영역: 층을 지날수록 한 칸이 보는 원본 구역이 넓어짐'입니다. 판 네 개가 왼쪽에서 오른쪽으로 화살표로 이어집니다. 판마다 위에서부터 판 제목, 회색 특징 맵 상자와 모양, 파란 '한 칸'이 있고, 그 아래 학습용 1번 그림(숫자 7)의 28 x 28 그림 위에 빨간 정사각형 구역이 겹쳐 있으며 '한 칸'에서 구역의 위 두 모서리로 빨간 선이 내려옵니다. 첫째 판 '첫 합성곱 뒤'는 '(16, 28, 28)'과 '3 x 3', 둘째 판 '첫 풀링 뒤'는 '(16, 14, 14)'와 '4 x 4', 셋째 판 '둘째 합성곱 뒤'는 '(32, 14, 14)'와 '8 x 8', 넷째 판 '둘째 풀링 뒤'는 '(32, 7, 7)'과 '10 x 10'이고, 빨간 구역은 그림에서 정확히 3, 4, 8, 10칸 너비입니다. 맨 아래 띠에 '마지막 특징 맵 (32, 7, 7)의 한 칸이 원본 28 x 28의 10 x 10 구역을 봄'이 적혀 있습니다.">


그 뒤 `nn.Linear`가 7 × 7 = 49칸을 모두 이어 그림 전체로 숫자를 정합니다.

## 5. 깊은 CNN: 배치 정규화와 ResNet

이 교안의 CNN은 합성곱 층이 두 개뿐이지만, 이미지 분류에 널리 쓰는 CNN은 층을 수십 개씩 쌓는 경우가 많습니다. 층을 깊게 쌓으면 앞쪽 층의 기울기가 작아지거나 층을 늘렸는데 오히려 학습 오차가 커질 수 있으므로, 그 원인과 깊은 CNN의 두 부품(배치 정규화, 잔차 연결), 두 부품으로 만든 ResNet을 차례로 봅니다.

### 5-1. 기울기 소실과 잔차 연결

**기울기 소실(vanishing gradient)**
- 층이 깊을수록 역전파가 층마다 기울기를 곱해 가는데, 곱하는 값의 크기가 1보다 작으면 앞쪽(입력 쪽) 층의 기울기가 0에 가깝게 작아져 그 층이 거의 학습되지 않는 현상입니다.
- 딥러닝 학습 원리 단원 교안 02에서 본 것처럼 역전파는 손실 쪽 층부터 기울기를 구해 앞 층으로 넘기며 곱하고, Sigmoid의 기울기는 가장 커도 0.25입니다. Sigmoid 층 10개를 지나면 활성화 함수의 기울기만 곱한 값은 가장 커도 다음과 같습니다.

$\displaystyle 0.25^{10} = \frac{1}{1{,}048{,}576} \approx 9.54 \times 10^{-7}$

- 가중치 쪽 곱이 이 감소를 메우지 못하면, 손실 쪽에서 넘어온 기울기가 첫 층에 도착할 때는 100만분의 1보다 작아집니다. 은닉층에 양수 쪽 기울기가 1인 ReLU를 많이 쓰는 까닭의 하나입니다.
- **기울기 폭주(exploding gradient):** 반대로 층마다 곱하는 값의 크기가 1보다 크면 기울기가 층을 지날수록 커져, 갱신이 지나치게 커지고 손실이 발산하는 현상입니다.
- **알아 두는 목적:** 층을 깊게 쌓았는데 앞쪽 층이 학습되지 않을 때 원인을 알고, 그 문제를 줄이는 구조(ReLU, 배치 정규화, 잔차 연결)를 고릅니다.

**잔차 연결(residual connection, skip connection)**
- 층 묶음의 입력 $x$를 그 묶음의 출력 $F(x)$에 그대로 더해 다음 층으로 보내는 연결입니다.

$\displaystyle y = F(x) + x$

- $F(x)$는 층 묶음(예: 합성곱 두 개)의 계산이고, $x$는 묶음을 건너뛰는 지름길로 전달됩니다. 더하려면 $F(x)$와 $x$의 모양이 같아야 합니다.
- **목적:** 층을 깊게 쌓을수록 오히려 학습 오차가 커지는 성능 저하(degradation)를 줄입니다(5-4).
- **기울기 전달:** 역전파 때 기울기가 지름길로 층을 건너뛰어 앞쪽 층까지 직접 전달됩니다. $y$를 $x$로 미분하면 더한 $x$ 쪽에서 1이 생기므로, $F'(x)$가 0에 가까워도 1이 남습니다([He et al., 2016, Identity Mappings in Deep Residual Networks](https://arxiv.org/abs/1603.05027)).

$\displaystyle \frac{dy}{dx} = F'(x) + 1$


<img src="images/67_vanishing_residual.png" width="1050" alt="제목은 '기울기 소실과 잔차 연결'입니다. 왼쪽 '깊게 쌓은 층' 판에서 흰 '입력' 상자, 파란 층 블록 네 개, 빨간 '손실' 상자가 회색 화살표로 이어지고, 그 아래 보라 기울기 화살표가 '손실' 쪽에서 '입력' 쪽으로 갈수록 가늘어져 마지막은 점선으로 사라집니다. 굵은 화살표 아래에 '기울기', 점선 아래에 '앞쪽 층: 기울기가 0에 가까움', 판 아래 상자에 '층마다 1보다 작은 값을 곱하면 기울기가 점점 작아짐'이 적혀 있습니다. 오른쪽 '잔차 연결' 판은 같은 줄에서 층 블록마다 남색 지름길 선이 블록을 건너뛰어 블록 뒤의 노란 '+' 원에서 블록 출력과 더해지고, 첫 지름길에 'x', 첫 블록에 'F(x)'가 적혀 있습니다. 보라 기울기 화살표는 '손실'에서 '입력'까지 같은 굵기로 이어지고, 그 아래에 '기울기'와 '앞쪽 층까지 기울기가 남음', 판 아래 상자에 'y = F(x) + x: 기울기가 지름길로 전달됨'이 적혀 있습니다.">


### 5-2. 배치 정규화

**배치 정규화(batch normalization, BatchNorm)**
- 미니배치마다 층 출력의 평균과 분산을 특징마다(합성곱 층에서는 채널마다) 계산해 평균 0, 분산 1로 표준화한 뒤, 학습하는 크기 $\gamma$를 곱하고 이동 $\beta$를 더하는 층입니다([Ioffe & Szegedy, 2015](https://arxiv.org/abs/1502.03167)).
- 채널 하나의 값을 $x$, 미니배치에서 계산한 그 채널의 평균과 분산을 $\mu_B$, $\sigma_B^2$라 하면 다음과 같습니다.

$\displaystyle \hat{x} = \frac{x - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}$

$\displaystyle y = \gamma \hat{x} + \beta$

- $\epsilon$은 분산이 0이어도 0으로 나누지 않게 더하는 작은 수입니다(PyTorch 기본 0.00001). $\gamma$는 1, $\beta$는 0에서 시작해 다른 파라미터처럼 역전파로 학습되므로, 평균 0, 분산 1이 맞지 않는 층은 크기와 위치를 다시 맞출 수 있습니다.
- **목적:** 층 출력의 크기를 미니배치마다 일정하게 맞춰 학습을 안정시킵니다. 원 논문은 배치 정규화로 훨씬 큰 학습률을 쓸 수 있었고, 이미지 분류 모델이 같은 정확도에 14분의 1의 학습 스텝으로 도달했다고 보고했습니다.
- **입력 표준화와의 관계:** 딥러닝 학습 원리 단원에서 입력 열을 평균 0, 표준편차 1로 맞춘 입력 표준화와 같은 생각을, 입력이 아니라 층 출력에 미니배치마다 적용합니다.
- **`nn.BatchNorm2d(채널 수)`:** 채널마다 (그림 수 × 높이 × 너비)개 값의 평균과 분산을 따로 계산합니다. $\gamma$(`weight`)와 $\beta$(`bias`)도 채널마다 하나씩입니다.
- **위치:** 합성곱 뒤, 활성화 함수 앞(`Conv2d` → `BatchNorm2d` → `ReLU`)에 두는 배치가 흔하고, ResNet도 이 자리에 둡니다. 트랜스포머는 배치 정규화 대신 주로 LayerNorm을 씁니다.


<img src="images/68_batch_norm.png" width="1050" alt="제목은 '배치 정규화: 채널마다 표준화한 뒤 γ, β로 조정'입니다. 윗줄에 번호 배지가 붙은 카드 세 장이 화살표로 이어지고, 카드마다 가로축 가운데 회색 점선 아래에 '0'이 있는 파란 히스토그램이 있습니다. 1 '층 출력' 카드는 넓은 분포가 점선보다 오른쪽에 있고 아래에 '미니배치의 평균과 분산 계산', 2 '표준화' 카드는 좁아진 분포가 점선 위에 가운데를 맞추고 아래에 '평균 0, 분산 1', 3 'γ 곱하고 β 더하기' 카드는 분포가 조금 넓어지고 오른쪽으로 조금 옮겨졌으며 아래에 '학습하는 크기 γ, 이동 β'가 적혀 있습니다. 아랫줄 왼쪽 '학습 모드 model.train()' 판에는 겹친 카드 묶음과 '이번 미니배치의 평균과 분산', 오른쪽 '평가 모드 model.eval()' 판에는 카드 한 장과 '학습 중 모은 이동 평균'이 있습니다.">


**학습 모드와 평가 모드**
- **학습 모드(`model.train()`):** 이번 미니배치의 평균과 분산으로 표준화하고(그래서 4-2에서 학습용의 마지막 작은 미니배치를 버립니다), 그 값의 이동 평균(`running_mean`, `running_var`)을 모아 둡니다. 미니배치마다 `이동 평균 = 0.9 * 이동 평균 + 0.1 * 이번 미니배치 값`으로 갱신합니다(기본 `momentum=0.1`).
- **평가 모드(`model.eval()`):** 모아 둔 이동 평균으로 표준화합니다. 같은 미니배치에 든 다른 그림에 따라 결과가 바뀌지 않아, 같은 그림은 늘 같은 예측을 냅니다.
- 그래서 `BatchNorm2d`가 든 모델은 학습 전에 `model.train()`, 평가 전에 `model.eval()`을 부릅니다. 교안 02에서 Dropout 때문에 학습 함수와 평가 함수에 이 두 줄을 더하는 것과 같은 이유입니다.

4-2에서 꺼낸 미니배치 64장을 첫 합성곱 층에 넣어 나온 특징 맵 (64, 16, 28, 28)에 배치 정규화를 적용하고, 학습 모드와 평가 모드의 출력을 비교합니다.

In [ ]:
# 첫 합성곱 층이 미니배치 64장에서 낸 특징 맵. 모양 (64, 16, 28, 28)
with torch.no_grad():
    conv_output = model[0](X_batch.to(device))

# 채널 16개의 배치 정규화 층. gamma는 weight(처음 1), beta는 bias(처음 0)입니다
batch_norm = nn.BatchNorm2d(16).to(device)

# 학습 모드: 이번 미니배치의 채널별 평균과 분산으로 표준화하고, 이동 평균을 한 번 갱신합니다
batch_norm.train()
with torch.no_grad():
    train_output = batch_norm(conv_output)

# 평가 모드: 모아 둔 이동 평균으로 표준화합니다
batch_norm.eval()
with torch.no_grad():
    eval_output = batch_norm(conv_output)

In [ ]:
# 채널마다 (그림 수, 높이, 너비) 세 축의 평균과 표준편차를 구해 앞 채널 4개를 봅니다
channel_dims = [0, 2, 3]
bn_table = pd.DataFrame({
    "합성곱 출력 평균": conv_output.mean(dim=channel_dims)[:4].tolist(),
    "합성곱 출력 표준편차": conv_output.std(dim=channel_dims)[:4].tolist(),
    "학습 모드 평균": train_output.mean(dim=channel_dims)[:4].tolist(),
    "학습 모드 표준편차": train_output.std(dim=channel_dims)[:4].tolist(),
    "평가 모드 평균": eval_output.mean(dim=channel_dims)[:4].tolist(),
    "평가 모드 표준편차": eval_output.std(dim=channel_dims)[:4].tolist(),
})
bn_table.index.name = "채널"
display(bn_table.round(4))

**표 읽기**
- **합성곱 출력:** 채널마다 평균(-0.3203~0.2933)과 표준편차(0.0782~0.3336)가 서로 다릅니다.
- **학습 모드:** 네 채널 모두 평균 0.0, 표준편차 0.9992~1.0000으로 맞춰졌습니다. $\gamma$는 1, $\beta$는 0에서 시작했으므로 표준화한 값이 그대로 나옵니다.
- **평가 모드:** 채널 0이 평균 -0.3037, 표준편차 0.0824로 합성곱 출력과 거의 같습니다. 이동 평균이 처음값(평균 0, 분산 1)에서 미니배치 한 번만큼만 옮겨 갔기 때문입니다. 같은 입력이어도 모드에 따라 출력이 다르므로 평가 전에 `model.eval()`을 부릅니다.

### 5-3. 잔차 블록

**잔차 블록(residual block)**
- 층 묶음 $F(x)$의 출력에 입력 $x$를 더하는 블록이고, ResNet은 이 블록을 반복해서 쌓습니다. ResNet-18·34는 합성곱 두 개짜리 기본 블록, ResNet-50 이상은 세 개짜리 병목 블록을 쓰며, 이 교안은 기본 블록을 만듭니다.
- **$F(x)$:** `Conv2d` → `BatchNorm2d` → `ReLU` → `Conv2d` → `BatchNorm2d`입니다. 합성곱은 3 × 3, `padding=1`이라 크기가 그대로입니다.
- **출력:** `ReLU(F(x) + x)`입니다.
- **모양이 바뀌는 블록:** 채널 수를 늘리거나 크기를 절반으로 줄이는 블록에서는 $F(x)$와 $x$의 모양이 달라 바로 더할 수 없으므로, 보통 지름길에 1 × 1 합성곱(크기도 절반으로 줄이면 stride 2)을 두어 $x$의 채널 수와 크기를 $F(x)$에 맞춘 뒤 더합니다.


<img src="images/69_residual_block.png" width="1050" alt="제목은 '잔차 블록: F(x)에 x를 더한 뒤 ReLU'입니다. 위 판 '같은 모양: x를 그대로 더함'에서는 '입력 x'(안에 '(16, 14, 14)')에서 'Conv2d 3 x 3', 'BatchNorm2d', 'ReLU', 'Conv2d 3 x 3', 'BatchNorm2d'를 지나 노란 '+' 원, 'ReLU', '출력'(안에 '(16, 14, 14)')으로 이어집니다. 다섯 상자 아래 중괄호에 'F(x)'가 있고, '입력 x'에서 남색 지름길 선이 다섯 상자 위를 건너 '+'에 닿으며 선 위에 'x'가 적혀 있습니다. 아래 판 '모양이 바뀜: 1 x 1 합성곱으로 x 모양을 맞춤'에서는 '입력 x'(안에 '(16, 14, 14)')에서 큰 상자 'F(x)'를 지나 '+', 회색 'ReLU', '출력'(안에 '(32, 7, 7)')으로 이어지고, 지름길 선이 '1 x 1 Conv2d, stride 2' 상자를 지나 '+'에 닿습니다.">


층을 차례로 통과시키는 `nn.Sequential`로는 입력을 따로 들고 있다가 더하는 계산을 쓸 수 없으므로, 파이썬 기초에서 배운 클래스 상속으로 `nn.Module`을 물려받아 블록을 만듭니다.

- **`__init__`:** 블록에 쓸 층을 만들어 `self`에 붙여 둡니다. 첫 줄에서 `super().__init__()`으로 물려받은 `nn.Module`의 준비를 먼저 합니다.
- **`forward`:** 입력이 들어왔을 때의 계산 순서를 씁니다. `block(features)`처럼 부르면 실행됩니다.

In [ ]:
class ResidualBlock(nn.Module):  # nn.Module을 물려받아 새 층 묶음을 만듭니다
    """합성곱 두 개의 출력 F(x)에 입력 x를 더한 뒤 ReLU를 지나는 잔차 블록입니다. 채널 수와 크기는 그대로입니다."""

    def __init__(self, channels):  # 블록을 만들 때 한 번 실행됩니다
        super().__init__()  # 물려받은 nn.Module의 준비를 먼저 합니다
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):  # block(x)처럼 부르면 실행됩니다
        # F(x): Conv2d -> BatchNorm2d -> ReLU -> Conv2d -> BatchNorm2d
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))

        # 입력 x를 더한 뒤 ReLU: 출력 = ReLU(F(x) + x)
        return torch.relu(out + x)

In [ ]:
# 블록의 합성곱 초깃값을 고정하고, 채널 16개짜리 잔차 블록을 만듭니다
torch.manual_seed(0)
block = ResidualBlock(16).to(device)

# CNN의 앞 세 층(합성곱, ReLU, 풀링)을 지난 특징 맵을 블록에 넣습니다. nn.Sequential은 리스트처럼 잘라 쓸 수 있습니다
with torch.no_grad():
    features = model[:3](X_batch.to(device))
    block_output = block(features)
print("블록 입력:", tuple(features.shape), "| 블록 출력:", tuple(block_output.shape))

# 블록의 파라미터 수(합성곱 2개, 배치 정규화 2개)
n_block_params = 0
for param in block.parameters():
    n_block_params += param.numel()
print("블록 파라미터 수:", n_block_params)

**결과 읽기**
- **블록 입력과 출력 (64, 16, 14, 14):** 모양이 같으므로 같은 블록을 몇 개든 이어서 쌓을 수 있습니다.
- **파라미터 4,704개:** 합성곱 두 개(16 × 16 × 3 × 3 + 16 = 2,320개씩)와 배치 정규화 두 개($\gamma$와 $\beta$ 16개씩, 32개씩)입니다.

### 5-4. ResNet

**ResNet(residual network)**
- 잔차 블록을 여러 개 쌓은 CNN입니다([He et al., 2016](https://arxiv.org/abs/1512.03385)). 첫 합성곱 다음에 잔차 블록을 단계별로 쌓고, 단계가 바뀔 때마다 채널 수를 두 배로 늘리고 크기를 절반으로 줄입니다.
- **끝 부분:** 마지막 특징 맵을 채널마다 평균 하나로 줄이는 전역 평균 풀링(global average pooling) 뒤에 `Linear`가 로짓을 냅니다. 이 교안의 CNN처럼 펼치지 않고 채널마다 숫자 하나만 남깁니다.


<img src="images/50_resnet18.png" width="1050" alt="제목은 'ResNet-18: 잔차 블록 8개를 쌓은 CNN'입니다. 윗줄은 평면 일러스트 고양이 얼굴 그림 '입력 그림 (3, 224, 224)'에서 'Conv2d 7 x 7', 'MaxPool2d'를 지나 단계 상자 '64채널', '128채널'로 이어지고, 아래로 굽은 화살표가 아랫줄 '256채널', '512채널', '전역 평균 풀링', 'Linear', 주황 막대그래프 '로짓 1000개'로 이어집니다. 단계 상자마다 지름길 곡선이 블록을 건너 '+' 원에 닿는 잔차 블록 아이콘 두 개가 있고, 단계가 바뀌는 화살표 세 개(64채널에서 128채널, 128채널에서 아래로 굽어 256채널, 256채널에서 512채널)에 '크기 절반'이 적혀 있습니다. 맨 아래 띠에 '합성곱 1층 + 잔차 블록 8개(합성곱 16층) + Linear 1층 = 18층'이 적혀 있습니다.">


- **깊이의 효과:** 논문의 ImageNet 실험에서 잔차 연결이 없는 CNN은 18층에서 34층으로 늘리자 검증 오차(top-1)가 27.94%에서 28.54%로 오히려 커졌고, 같은 구조에 잔차 연결을 넣은 ResNet은 27.88%에서 25.03%로 줄었습니다. 34층 CNN은 학습 오차도 18층보다 컸고, 논문은 배치 정규화를 써서 기울기가 사라지지는 않았다고 확인했습니다. 논문은 이 문제를 성능 저하(degradation problem)라 부르고 잔차 연결로 줄였습니다.
- **152층:** 152층 ResNet 두 개를 포함해 깊이가 다른 ResNet 여섯 개를 합친 앙상블로 ILSVRC 2015 이미지 분류 대회 1위(top-5 오차 3.57%)를 했습니다.
- **쓰임:** `torchvision` 라이브러리에 `torchvision.models.resnet18`, `resnet50`처럼 층 구성이 미리 정의된 ResNet이 있습니다. 트랜스포머도 층마다 잔차 연결을 씁니다.

## 6. 학습하기

모델이 준비됐으므로, 지난 시간과 같은 손실과 `train_one_epoch`로 CNN을 학습합니다.

### 6-1. 10에폭 학습하기

- **설정:** Adam 학습률 0.001, 배치 크기 64, 10에폭입니다. 지난 시간 DNN과 같은 설정이고(학습용 `drop_last=True`만 더함) 모델만 CNN으로 바꿨습니다.
- **기록:** 에폭마다 학습 손실과 학습 데이터 정확도를 기록합니다. 학습 데이터 정확도는 학습용 10,000장을 한 번에 모델에 넣어 argmax 예측으로 계산합니다(지난 시간과 같습니다). 테스트 데이터는 학습을 마친 뒤 한 번만 씁니다.
- **실행 환경:** 계산 장치와 스레드 수에 따라 소수 셋째 자리, 틀린 그림의 수와 번호, 필터 번호가 달라질 수 있습니다. 결론은 곡선의 모양과 큰 차이로 읽습니다.
- **학습 전 기록:** 학습 전·후를 비교하려고, 학습을 시작하기 전에 첫 층 필터와 테스트 그림 10장의 예측을 저장해 둡니다.
- CPU에서 에폭마다 몇 초씩 걸립니다.

학습 함수와 평가 함수는 지난 시간에 만든 것을 그대로 씁니다.

In [ ]:
# 지난 시간에 만든 학습 함수와 평가 함수를 그대로 씁니다
def train_one_epoch(model, loader, loss_fn, optimizer, scheduler=None):
    """미니배치마다 파라미터를 한 번씩 갱신하고, 그 에폭 미니배치 손실의 평균을 돌려줍니다."""
    total_loss = 0.0
    for X_batch, y_batch in loader:
        # 미니배치를 모델이 있는 계산 장치로 옮깁니다
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # 지난 스텝의 기울기를 지우고, 순전파 -> 손실 -> 역전파 -> 갱신
        optimizer.zero_grad()
        loss = loss_fn(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()

        # 스케줄러가 있으면 스텝마다 학습률을 바꿉니다
        if scheduler is not None:
            scheduler.step()

        total_loss += loss.item()

    # len(loader)는 미니배치 수입니다
    return total_loss / len(loader)


def evaluate(model, loader, loss_fn):
    """기울기 없이 예측해 (평균 손실, 정확도)를 돌려줍니다."""
    total_loss = 0.0
    correct = 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            logits = model(X_batch)

            # 마지막 미니배치는 장 수가 적으므로, 손실에 장 수를 곱해 더한 뒤 전체 장 수로 나눕니다
            total_loss += loss_fn(logits, y_batch).item() * len(y_batch)
            correct += (logits.argmax(dim=1) == y_batch).sum().item()

    n_samples = len(loader.dataset)
    return total_loss / n_samples, correct / n_samples

In [ ]:
# 지난 시간 다중 분류와 같은 교차 엔트로피 손실. 교차 엔트로피 = 평균(-log(정답 칸의 확률))
loss_fn = nn.CrossEntropyLoss()

# 학습 전 상태에서 시작하도록 새 모델과 옵티마이저를 만듭니다
torch.manual_seed(0)  # 모델 초깃값과 미니배치 섞는 순서를 고정합니다
model = make_cnn()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# 학습 전 첫 층 필터와 테스트 그림 앞 10장의 예측을 저장해 둡니다
filters_before = model[0].weight.detach().cpu().clone()
with torch.no_grad():
    predictions_before = model(X_test[:10].to(device)).argmax(dim=1).cpu()

# 에폭마다 학습하고, 학습 데이터 정확도를 기록합니다
history_rows = []
for epoch in range(1, 11):
    train_loss = train_one_epoch(model, train_loader, loss_fn, optimizer)

    # 학습용 10,000장을 한 번에 넣어 예측하고 정확도를 구합니다
    with torch.no_grad():
        train_predictions = model(X_train.to(device)).argmax(dim=1).cpu()
    train_accuracy = (train_predictions == y_train).sum().item() / len(y_train)
    history_rows.append({"에폭": epoch, "학습 손실": train_loss, "학습 데이터 정확도": train_accuracy})

# 행 번호가 에폭이 되게 표를 만들고 소수 넷째 자리까지 봅니다
history_table = pd.DataFrame(history_rows).set_index("에폭")
display(history_table.round(4))

**결과 읽기**
- **학습 손실:** 1에폭 0.7925에서 10에폭 0.0367로 줄었습니다. 첫 에폭에 가장 많이 줄고, 뒤로 갈수록 천천히 줄어듭니다.
- **학습 데이터 정확도:** 1에폭 뒤 0.9107에서 10에폭 뒤 0.9939로 올랐습니다. 같은 설정의 지난 시간 DNN은 10에폭 뒤 0.9879였습니다.

### 6-2. 테스트 데이터로 한 번 확인하기

학습 데이터 정확도는 모델이 이미 본 그림의 결과이므로, 처음 보는 테스트 2,000장으로 한 번 평가합니다.

In [ ]:
# 학습을 마친 모델을 테스트 데이터로 한 번 평가합니다
test_loss, test_accuracy = evaluate(model, test_loader, loss_fn)
print("테스트 손실:", round(test_loss, 4), "| 테스트 정확도:", round(test_accuracy, 4))

**결과 읽기**
- **테스트 정확도 0.981:** 2,000장 가운데 38장만 틀렸습니다. 같은 그림, 같은 설정에서 지난 시간 DNN은 0.948(104장 틀림)이었습니다.
- **틀린 비율:** 학습 데이터는 약 0.6%(정확도 0.9939), 테스트 데이터는 약 1.9%입니다. 이 차이가 과적합인지는 교안 02에서 검증 데이터로 가립니다.

같은 테스트 그림 10장을 학습 전 모델과 학습 후 모델이 각각 어떻게 예측했는지 나란히 봅니다.

In [ ]:
# 위 줄은 학습 전 예측, 아래 줄은 학습 후 예측. 맞히면 초록, 틀리면 빨간 제목입니다
with torch.no_grad():
    predictions_after = model(X_test[:10].to(device)).argmax(dim=1).cpu()
prediction_rows = {"학습 전": predictions_before, "학습 후": predictions_after}

plt.figure(figsize=(14, 3.8))
for row, row_name in enumerate(prediction_rows):
    for k in range(10):
        plt.subplot(2, 10, row * 10 + k + 1)
        plt.imshow(X_test[k, 0], cmap="gray_r")

        # 맞혔는지에 따라 제목 색을 정합니다
        predicted = prediction_rows[row_name][k]
        if predicted == y_test[k]:
            title_color = "green"
        else:
            title_color = "red"
        plt.title(f"{row_name} {class_names[predicted]}", fontsize=12, color=title_color)
        plt.axis("off")
plt.tight_layout()
plt.show()

**그래프 읽기**
- **학습 전:** 무작위 파라미터라 10장을 모두 0으로 예측해, 정답이 0인 1장만 맞혔습니다.
- **학습 후:** 정답 1을 3으로 읽은 1장만 틀리고 9장을 맞혔습니다.

## 7. 필터, 특징 맵, 틀린 그림

정확도 숫자만으로는 모델이 어떤 특징으로 맞히는지 알 수 없으므로, 학습한 필터와 특징 맵, 틀린 그림을 그림으로 봅니다.

### 7-1. 첫 층의 필터 16개

첫 층 `model[0]`의 `weight`는 (16, 1, 3, 3), 학습으로 정해진 3 × 3 필터 16개입니다. 앞에서 손으로 정한 필터처럼, 빨간 칸(양수)에 획이 겹치고 파란 칸(음수)이 비어 있을 때 큰 값을 냅니다. 학습이 필터를 어떻게 바꿨는지 보려고, 학습 전에 저장한 필터와 같은 색 범위로 나란히 그립니다.

In [ ]:
# 학습한 첫 층 필터 16개(16, 1, 3, 3)를 CPU로 옮기고 기울기 기록을 뗍니다
filters_learned = model[0].weight.detach().cpu()
print("가장 큰 절댓값: 학습 전", round(filters_before.abs().max().item(), 4),
      "| 학습 후", round(filters_learned.abs().max().item(), 4))

# 두 줄이 같은 색 범위를 쓰고, 0이 흰색이 되도록 양쪽을 같게 둡니다
color_limit = max(filters_before.abs().max().item(), filters_learned.abs().max().item())

# 위 줄은 학습 전, 아래 줄은 학습 후 필터. 열 번호가 필터 번호입니다
filter_rows = {"학습 전": filters_before, "학습 후": filters_learned}
plt.figure(figsize=(14, 2.8))
for row, row_name in enumerate(filter_rows):
    for k in range(16):
        plt.subplot(2, 16, row * 16 + k + 1)
        plt.imshow(filter_rows[row_name][k, 0], cmap="bwr", vmin=-color_limit, vmax=color_limit)
        plt.xticks([])
        plt.yticks([])
        if row == 0:
            plt.title(f"{k}", fontsize=11)
        if k == 0:
            plt.ylabel(row_name, fontsize=11)
plt.suptitle("첫 층 필터 16개의 학습 전과 학습 후 (빨강: 양수, 파랑: 음수)", fontsize=14)
plt.tight_layout()
plt.show()

**그래프 읽기**
- **학습 후 색이 진해짐:** 가장 큰 절댓값이 0.3325에서 0.6323으로 약 2배가 됐습니다.
- **무늬는 학습 전과 대체로 비슷함:** 이번 학습은 무작위 초깃값에 있던 빨강·파랑 배치를 크게 바꾸지 않고 더 진하게 만들었습니다.
- **경계를 찾는 필터:** 한쪽이 빨갛고 반대쪽이 파란 필터는 그 방향의 경계(획이 시작하거나 끝나는 자리)에서 큰 값을 냅니다.
- **예:** 필터 11은 학습 전에도 위가 붉고 아래가 푸르스름했고, 학습 후 위 행 빨강, 아래 행 파랑이 뚜렷해져 가로 획의 아래쪽 경계를 찾습니다. 필터 14는 왼쪽 열이 파래져 획의 왼쪽 경계를 찾습니다.

### 7-2. 숫자 하나의 특징 맵

같은 필터 16개가 테스트 그림 한 장에서 무엇을 찾았는지, 첫 합성곱 묶음의 ReLU까지 통과한 특징 맵 16장을 봅니다.

- **`model[:2]`:** `nn.Sequential`도 리스트처럼 앞 두 층(합성곱, ReLU)만 잘라 쓸 수 있습니다.

In [ ]:
# 테스트 0번 그림 한 장을 첫 합성곱과 ReLU까지만 통과시킵니다. 모양 (1, 16, 28, 28)
sample = X_test[0:1].to(device)
with torch.no_grad():
    sample_maps = model[:2](sample).cpu()
print("정답:", y_test[0].item(), "| 특징 맵 모양:", tuple(sample_maps.shape))

# 첫 칸은 원본, 나머지 16칸은 필터마다의 특징 맵(진할수록 큰 값)
plt.figure(figsize=(14, 7.5))
plt.subplot(3, 6, 1)
plt.imshow(X_test[0, 0], cmap="gray_r")
plt.title("원본", fontsize=12)
plt.axis("off")
for k in range(16):
    plt.subplot(3, 6, k + 2)
    plt.imshow(sample_maps[0, k], cmap="gray_r")
    plt.title(f"필터 {k}", fontsize=12)
    plt.axis("off")
plt.tight_layout()
plt.show()

**그래프 읽기**
- **필터마다 다른 획:** 숫자 2 그림에서 필터 11의 맵은 가로 방향 획(아래 곡선, 위 획)만, 필터 14의 맵은 비스듬히 내려오는 획만 남깁니다. 위에서 본 학습 후 필터 모양과 같은 방향입니다.
- **획 전체가 진한 맵:** 필터 7, 15처럼 원본과 거의 같은 맵도 있습니다. 배경이 회색인 맵(필터 2, 9, 10, 12)은 절편이 양수라 빈 곳에서도 0보다 큰 값이 나온 것입니다.
- 둘째 합성곱 묶음의 필터는 이 16장을 한꺼번에 보고 그 출력 한 칸의 수용 영역이 원본 8 × 8이므로(4-3), 여러 획을 묶은 더 큰 모양을 찾을 수 있습니다.

### 7-3. 틀린 그림과 헷갈린 짝

어떤 그림을 틀렸고 어떤 숫자끼리 헷갈렸는지 보면 모델의 약점을 알 수 있으므로, 테스트 그림의 예측을 모아 틀린 그림과 혼동 행렬을 만듭니다.

- **혼동 행렬(confusion matrix):** 행은 정답, 열은 예측인 10 × 10 표입니다. 대각선(정답 = 예측)은 맞힌 수, 나머지 칸은 그 짝으로 틀린 수입니다. 예: 4행 9열이 5면 정답 4를 9로 예측한 그림이 5장입니다.

In [ ]:
# 테스트 그림 2,000장의 예측(로짓이 가장 큰 칸)을 구합니다
with torch.no_grad():
    test_predictions = model(X_test.to(device)).argmax(dim=1).cpu()

# 틀린 그림의 번호를 모읍니다
wrong_index = []
for i in range(len(y_test)):
    if test_predictions[i] != y_test[i]:
        wrong_index.append(i)
print("틀린 그림:", len(wrong_index), "/", len(y_test))

In [ ]:
# 틀린 그림 앞 10장을 정답과 예측으로 그립니다
plt.figure(figsize=(14, 2.6))
for k in range(10):
    i = wrong_index[k]
    plt.subplot(1, 10, k + 1)
    plt.imshow(X_test[i, 0], cmap="gray_r")
    plt.title(f"정답 {class_names[y_test[i]]}\n예측 {class_names[test_predictions[i]]}", fontsize=12)
    plt.axis("off")
plt.show()

In [ ]:
# 혼동 행렬: 행은 정답, 열은 예측. 그림 한 장마다 해당 칸에 1을 더합니다
confusion = torch.zeros(10, 10, dtype=torch.long)
for i in range(len(y_test)):
    confusion[y_test[i], test_predictions[i]] += 1
confusion_table = pd.DataFrame(confusion.tolist(), index=class_names, columns=class_names)
display(confusion_table)

# 대각선(맞힌 수)을 0으로 바꾼 뒤 가장 큰 칸이 가장 많이 헷갈린 짝입니다. 같은 값이면 모두 출력합니다
off_diagonal = confusion - torch.diag(torch.diag(confusion))
top_count = off_diagonal.max().item()
for true_label in range(10):
    for predicted_label in range(10):
        if off_diagonal[true_label, predicted_label] == top_count:
            print(f"가장 많이 헷갈린 짝: 정답 {class_names[true_label]}, 예측 {class_names[predicted_label]}인 그림 {top_count}장")

**결과 읽기**
- **틀린 그림 38장:** 앞 10장 가운데 5장이 정답 6이고, 2장은 0으로, 3장은 5로 예측했습니다. 획이 짧거나 고리가 열린 6, 옆에 점이 있는 1처럼 흔히 쓰는 모양과 다르게 쓴 그림이 많습니다.
- **혼동 행렬:** 대각선 밖은 대부분 0~2장입니다. 가장 많이 헷갈린 짝은 정답 6을 0으로, 5로 예측한 각 4장입니다.
- 특정 짝에 틀린 그림이 몰려 있으면, 그 숫자를 쓴 그림을 더 모으거나 그 두 숫자를 가르는 획을 모델이 보는지 특징 맵으로 확인하는 것이 다음 할 일입니다.

## 8. 모델 저장과 불러오기

학습한 모델을 서비스에 쓰거나 다음 날 이어서 쓰려면 매번 다시 학습할 수 없으므로, 학습한 파라미터를 파일로 저장했다가 불러옵니다.

- **`model.state_dict()`:** 파라미터 이름(`0.weight` 등)과 값(텐서)을 짝지은 딕셔너리입니다. 모델의 구조(어떤 층을 어떤 순서로 쌓았는지)는 들어 있지 않습니다.
- **`torch.save(딕셔너리, 경로)`:** 텐서가 든 딕셔너리를 파일로 저장합니다. 파일 이름 끝은 보통 `.pt`입니다.
- **`load_state_dict(딕셔너리)`:** 같은 구조로 새로 만든 모델에 저장한 값을 넣습니다. 그래서 불러올 때도 `make_cnn()`으로 같은 구조를 먼저 만듭니다.
- **`torch.load(경로, map_location=device)`:** 파일의 텐서를 현재 계산 장치로 옮겨 읽습니다. GPU에서 저장한 파일을 CPU에서 읽을 때도 이 인자로 읽습니다.

In [ ]:
# output 폴더는 저장소에 올리지 않는 폴더라, 처음 실행할 때 만듭니다
output_dir = material_dir / "output"
output_dir.mkdir(exist_ok=True)

# 학습한 파라미터를 파일로 저장합니다
model_path = output_dir / "mnist_cnn.pt"
torch.save(model.state_dict(), model_path)
print("저장한 파일:", model_path.name, "|", round(model_path.stat().st_size / 1024, 1), "KB")

In [ ]:
# 같은 구조의 새 모델: 파라미터가 무작위라 아직 숫자를 구별하지 못합니다
loaded_model = make_cnn()
_, before_accuracy = evaluate(loaded_model, test_loader, loss_fn)
print("불러오기 전 테스트 정확도:", round(before_accuracy, 4))

# 저장한 파일을 읽어 새 모델에 넣습니다
loaded_model.load_state_dict(torch.load(model_path, map_location=device))
_, after_accuracy = evaluate(loaded_model, test_loader, loss_fn)
print("불러온 뒤 테스트 정확도:", round(after_accuracy, 4))

# 불러온 모델의 예측이 학습한 모델의 예측과 2,000장 모두 같은지 확인합니다
with torch.no_grad():
    loaded_predictions = loaded_model(X_test.to(device)).argmax(dim=1).cpu()
print("예측이 모두 같은가:", torch.equal(loaded_predictions, test_predictions))

**결과 읽기**
- **불러오기 전:** 무작위 파라미터라 정확도가 무작위로 고를 때의 정확도(약 0.1)와 같은 수준입니다.
- **불러온 뒤:** 학습을 마친 모델과 같은 0.981이고, 2,000장의 예측이 모두 같습니다. 파일 크기는 83.0KB입니다. 파라미터 20,490개 × 4바이트(float32) = 약 80KB에 파라미터 이름 같은 정보가 더해진 크기입니다.
- **실무 판단:** 파일에는 구조가 없으므로, 모델을 만드는 코드(`make_cnn`)도 함께 보관해야 다른 컴퓨터에서 불러올 수 있습니다.

## 9. 🖐️ 함께 따라하기: 옷 사진 분류

본실습의 `make_cnn()`과 학습 함수를 옷 사진에 적용해, 같은 CNN이 손글씨가 아닌 그림에서도 작동하는지와 어떤 옷끼리 헷갈리는지 확인하고, 5절의 배치 정규화와 잔차 블록을 CNN에 넣으면 학습이 어떻게 달라지는지를 직접 작성해 확인합니다.

**본실습과 다른 점**
- **데이터:** 옷 사진 [Fashion-MNIST](https://github.com/zalandoresearch/fashion-mnist)(Xiao, Rasul & Vollgraf, 2017, MIT 라이선스)에서 고른 학습용 10,000장과 테스트용 2,000장입니다.
- **입력 크기:** 28 × 28 흑백, 종류 10개로 MNIST와 같아 `make_cnn()`을 그대로 씁니다.
- **읽는 법:** 숫자보다 겉모양이 비슷한 종류가 많아, 혼동 행렬에서 어떤 옷끼리 헷갈리는지 봅니다.
- **모델 비교:** 9-5와 9-6에서 `make_cnn()`에 배치 정규화와 잔차 블록을 차례로 넣은 CNN을 학습해, 원래 CNN과 학습 손실을 비교합니다.

**실습 진행 방법:** 작성 셀의 `...`만 주석에 맞춰 채웁니다. 완성 코드는 `정답/교안_01_정답.ipynb`의 같은 절에 있습니다.

**설정**
- 세 모델 모두 배치 크기 64(학습용만 섞고 마지막 작은 미니배치는 버리기), Adam 학습률 0.001, 10에폭, 모델을 만들기 직전 `torch.manual_seed(0)`
- 배치 정규화가 든 모델은 학습 전에 `model.train()`, 예측과 평가 전에 `model.eval()`을 부릅니다(5-2).
- 테스트 데이터는 학습을 마친 모델마다 한 번만 씁니다. 어떤 모델을 고를지는 교안 02에서 검증 데이터로 정합니다.

### 9-1. 데이터 읽기

옷 사진 10종류를 한 장씩 봅니다.

In [ ]:
# 옷 사진: 그림과 정답 텐서 4개, 종류 이름 10개
fashion = torch.load(data_dir / "따라하기" / "fashion_small.pt")
fashion_names = fashion["class_names"]
print("학습용:", fashion["train_images"].shape, "| 테스트용:", fashion["test_images"].shape)
print("종류:", fashion_names)

# 종류마다 학습용 첫 그림 한 장씩 그립니다
plt.figure(figsize=(14, 2.6))
for label in range(10):
    # 그 종류가 처음 나오는 학습용 그림 번호를 앞에서부터 찾습니다
    first_index = 0
    while fashion["train_labels"][first_index] != label:
        first_index += 1
    plt.subplot(1, 10, label + 1)
    plt.imshow(fashion["train_images"][first_index], cmap="gray_r")
    plt.title(fashion_names[label], fontsize=12)
    plt.axis("off")
plt.show()

### 9-2. CNN 입력 모양과 DataLoader

In [ ]:
# 🖐️ 함께 따라하기: 그림은 0~1 실수로 바꾸고 채널 축을 끼워 (N, 1, 28, 28)로, 정답은 그대로 꺼내세요
X_fashion_train = ...
X_fashion_test = ...
y_fashion_train = ...
y_fashion_test = ...
print("X_fashion_train:", X_fashion_train.shape, X_fashion_train.dtype, "| y_fashion_train:", y_fashion_train.shape)

# 🖐️ 학습용과 테스트용 DataLoader를 만드세요(배치 크기 64, 학습용만 섞고 마지막 작은 미니배치는 버립니다)
fashion_train_loader = ...
fashion_test_loader = ...
print("학습 미니배치 수:", len(fashion_train_loader), "| 테스트 미니배치 수:", len(fashion_test_loader))

### 9-3. CNN 만들고 학습하기

본실습의 `make_cnn()`, `loss_fn`, `train_one_epoch`, `evaluate`를 그대로 씁니다.

In [ ]:
# 🖐️ 함께 따라하기: 본실습과 같은 CNN과 Adam 옵티마이저(학습률 0.001)를 만드세요
torch.manual_seed(0)  # 모델 초깃값과 미니배치 섞는 순서를 고정합니다
fashion_model = ...
fashion_optimizer = ...

# 🖐️ 10에폭 학습하며 에폭마다 학습 손실과 학습 데이터 정확도를 기록하세요
fashion_rows = []
for epoch in range(1, 11):
    train_loss = ...

    # 🖐️ 학습용 사진 전체를 한 번에 넣어 예측(argmax)하세요
    with torch.no_grad():
        train_predictions = ...
    train_accuracy = (train_predictions == y_fashion_train).sum().item() / len(y_fashion_train)
    fashion_rows.append({"에폭": epoch, "학습 손실": train_loss, "학습 데이터 정확도": train_accuracy})

# 행 번호가 에폭이 되게 표를 만들고 소수 넷째 자리까지 봅니다
fashion_history = pd.DataFrame(fashion_rows).set_index("에폭")
display(fashion_history.round(4))

### 9-4. 테스트 정확도와 혼동 행렬

학습을 마친 모델을 테스트 데이터로 한 번 평가하고, 본실습 7-3처럼 혼동 행렬로 헷갈린 짝을 봅니다.

In [ ]:
# 🖐️ 함께 따라하기: 테스트 데이터로 한 번 평가하고, 테스트 사진 2,000장의 예측(argmax)을 구하세요
fashion_test_loss, fashion_test_accuracy = ...
with torch.no_grad():
    fashion_predictions = ...
print("테스트 정확도:", round(fashion_test_accuracy, 4))

# 🖐️ 혼동 행렬을 만드세요(행은 정답, 열은 예측). 0으로 채운 10 x 10 표에서 시작합니다
fashion_confusion = ...
for i in range(len(y_fashion_test)):
    ...
fashion_table = pd.DataFrame(fashion_confusion.tolist(), index=fashion_names, columns=fashion_names)
display(fashion_table)

# 사진 2,000장이 한 칸씩 들어갔다면 표의 합은 2000입니다
print("표의 합:", fashion_confusion.sum().item())

In [ ]:
# 종류마다 맞힌 비율 = 대각선 칸 / 그 행(정답)의 합
class_rows = []
for label in range(10):
    class_rows.append({"종류": fashion_names[label],
                       "맞힌 비율": fashion_confusion[label, label].item() / fashion_confusion[label].sum().item()})
class_table = pd.DataFrame(class_rows)

plt.figure(figsize=(14, 4))
plt.bar(class_table["종류"], class_table["맞힌 비율"], color="steelblue")
for k in range(10):
    ratio = class_table["맞힌 비율"][k]
    plt.text(k, ratio + 0.02, f"{ratio:.2f}", ha="center", fontsize=11)  # 막대 위에 비율을 적습니다
plt.xticks(fontsize=12)
plt.yticks(fontsize=11)
plt.ylim(0, 1.1)
plt.ylabel("맞힌 비율", fontsize=12)
plt.title("옷 종류마다 맞힌 비율(테스트)", fontsize=14)
plt.show()

**확인 퀴즈**: 옷 사진 CNN에서 맞힌 비율이 가장 낮은 종류는 무엇이고, 무엇과 헷갈릴까요? 맞힌 비율 막대와 혼동 행렬로 설명해 봅니다.

<details><summary>정답 보기</summary>

테스트 정확도는 약 0.87로 손글씨(0.981)보다 낮습니다.

- **약한 종류:** 스웨터가 약 0.68로 가장 낮고, 셔츠(약 0.76)와 코트(약 0.77)가 다음입니다.
- **헷갈린 짝:** 이 셋과 티셔츠 사이에 틀린 그림이 16~37장씩 몰려 있습니다(스웨터를 셔츠로, 코트를 셔츠로, 셔츠를 티셔츠로). 윗옷이 아닌 짝으로는 앵클부츠를 운동화로 14장 틀렸습니다.
- **이유:** 28 × 28 흑백 사진에서는 깃, 단추, 소매 길이 같은 작은 차이가 몇 픽셀뿐이라, 겉모양이 비슷한 윗옷끼리 구별하기 어렵습니다. 바지·운동화는 0.96 넘게 맞힙니다.
- **실무 판단:** 헷갈리는 종류의 사진을 더 모으거나, 해상도가 높은 사진을 쓰는 것이 다음 할 일입니다. 혼동 행렬에서 큰 칸을 먼저 확인합니다.

</details>

### 9-5. 배치 정규화를 넣은 CNN

`make_cnn()`의 두 합성곱 뒤, ReLU 앞에 `nn.BatchNorm2d`를 넣습니다(5-2의 위치). 나머지 층과 설정, 시드는 9-3과 같게 두어 배치 정규화만 다르게 합니다.

- **넣는 층:** `nn.BatchNorm2d(16)`, `nn.BatchNorm2d(32)`입니다. 채널 수는 바로 앞 합성곱의 출력 채널 수와 같습니다.
- **파라미터 수:** `model.parameters()`가 돌려주는 텐서마다 `numel()`(원소 수)을 더합니다. 5-3에서 블록의 파라미터를 센 방법과 같습니다.

In [ ]:
# 🖐️ 함께 따라하기: 모델의 파라미터 수를 돌려주는 함수를 완성하세요(파라미터 텐서마다 numel()을 더합니다)
def count_parameters(model):
    """모델의 파라미터(가중치, 절편, 배치 정규화의 gamma와 beta) 개수를 모두 더해 돌려줍니다."""
    total = 0
    for param in model.parameters():
        total += ...
    return total


# 🖐️ make_cnn()의 합성곱 뒤마다 BatchNorm2d를 넣은 CNN을 만드세요(Conv2d -> BatchNorm2d -> ReLU -> MaxPool2d)
torch.manual_seed(0)  # 9-3과 같은 seed
bn_model = nn.Sequential(
    ...,  # 🖐️ 합성곱 묶음 1: Conv2d(1, 16), BatchNorm2d(16), ReLU, MaxPool2d(2) -> (16, 14, 14)
    ...,  # 🖐️ 합성곱 묶음 2: Conv2d(16, 32), BatchNorm2d(32), ReLU, MaxPool2d(2) -> (32, 7, 7)
    nn.Flatten(),
    nn.Linear(32 * 7 * 7, 10),
).to(device)
print("파라미터 수: 원래 CNN", count_parameters(fashion_model), "| 배치 정규화 CNN", count_parameters(bn_model))

9-3의 학습 반복에 학습 모드와 평가 모드 전환을 더해 함수로 만듭니다. 9-6의 모델도 이 함수로 학습합니다.

In [ ]:
# 🖐️ 함께 따라하기: 9-3의 학습 반복에 학습 모드와 평가 모드 전환을 더한 함수를 완성하세요
def fit_fashion(model, optimizer):
    """옷 사진으로 10에폭 학습하고, 에폭마다 학습 손실과 학습 데이터 정확도를 기록한 표를 돌려줍니다."""
    rows = []
    for epoch in range(1, 11):
        # 🖐️ 학습 모드로 바꾼 뒤 train_one_epoch로 한 에폭 학습하세요(배치 정규화가 미니배치의 평균과 분산을 씁니다)
        ...
        train_loss = ...

        # 🖐️ 평가 모드로 바꾼 뒤 학습용 사진 전체의 예측(argmax)을 구하세요(배치 정규화가 이동 평균을 씁니다)
        ...
        with torch.no_grad():
            train_predictions = ...
        train_accuracy = (train_predictions == y_fashion_train).sum().item() / len(y_fashion_train)
        rows.append({"에폭": epoch, "학습 손실": train_loss, "학습 데이터 정확도": train_accuracy})
    return pd.DataFrame(rows).set_index("에폭")


# 🖐️ Adam 옵티마이저(학습률 0.001)를 만들고 fit_fashion으로 10에폭 학습하세요
bn_optimizer = ...
bn_history = ...
display(bn_history.round(4))

In [ ]:
# 🖐️ 함께 따라하기: 평가 모드로 바꾼 뒤 테스트 데이터로 한 번 평가하세요
...
bn_test_loss, bn_test_accuracy = ...


def compare_row(name, model, history, test_accuracy):
    """비교 표의 한 행: 파라미터 수, 1에폭과 10에폭의 학습 손실, 테스트 정확도."""
    return {"모델": name, "파라미터 수": count_parameters(model),
            "1에폭 학습 손실": history.loc[1, "학습 손실"], "10에폭 학습 손실": history.loc[10, "학습 손실"],
            "테스트 정확도": test_accuracy}


# 9-3의 원래 CNN과 나란히 봅니다
compare_rows = [compare_row("원래 CNN", fashion_model, fashion_history, fashion_test_accuracy),
                compare_row("배치 정규화 CNN", bn_model, bn_history, bn_test_accuracy)]
display(pd.DataFrame(compare_rows).set_index("모델").round(4))

**결과 읽기**
- **파라미터 20,490개 → 20,586개:** 배치 정규화 두 층의 $\gamma$와 $\beta$가 채널마다 하나씩, (16 + 32) × 2 = 96개 늘었습니다. 이동 평균(`running_mean`, `running_var`)은 학습하는 값이 아니라 `parameters()`에 들지 않습니다.
- **학습 손실:** 1에폭 0.9679 → 0.5968, 10에폭 0.2848 → 0.1609로 배치 정규화 CNN이 낮습니다. 4에폭(0.2689)에 이미 원래 CNN의 10에폭 손실보다 낮아졌습니다.
- **테스트 정확도 0.8705 → 0.8805:** 0.010 차이는 2,000장 가운데 20장으로 시드만 바꿔도 달라질 수 있는 크기라, 이 결과로 배치 정규화가 테스트 정확도를 올렸다고 말할 수는 없습니다. 같은 에폭에서 학습 손실을 더 빨리 줄인 것이 이번에 확인한 차이입니다.

### 9-6. 잔차 블록을 넣은 CNN

5-3의 `ResidualBlock`(채널 16개) 하나를 배치 정규화 CNN의 첫 풀링 뒤에 넣습니다. 블록은 입력과 출력이 같은 모양 (16, 14, 14)이므로 앞뒤 층을 바꾸지 않고 끼울 수 있습니다. 클래스는 5-3에서 만든 것을 그대로 씁니다.

In [ ]:
# 🖐️ 함께 따라하기: 배치 정규화 CNN의 첫 풀링 뒤에 5-3의 ResidualBlock(16)을 하나 넣으세요
torch.manual_seed(0)  # 9-3과 같은 seed
res_model = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),   # (16, 14, 14)
    ...,                                                                                           # 🖐️ 잔차 블록: (16, 14, 14) 그대로
    nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),  # (32, 7, 7)
    nn.Flatten(),
    nn.Linear(32 * 7 * 7, 10),
).to(device)

# 🖐️ 잔차 블록(4번 층)과 모델 전체의 파라미터 수를 count_parameters로 구하세요
print("잔차 블록:", ..., "| 잔차 블록 CNN 전체:", ...)

**결과 읽기**
- **파라미터:** 블록 4,704개는 5-3의 블록과 같고, 전체 25,290개는 배치 정규화 CNN(20,586개)에 블록을 더한 수입니다.
- **수용 영역:** 블록의 3 × 3 합성곱 두 개가 첫 풀링 뒤에 들어가는데, 풀링 뒤 맵은 이웃 칸이 원본에서 2칸씩 떨어져 있어 합성곱 하나마다 원본 기준 4칸씩 넓어지므로, 마지막 특징 맵 가운데 칸의 수용 영역이 10 × 10에서 18 × 18로 넓어집니다(4 → 8 → 12 → 16 → 18).

같은 `fit_fashion`으로 10에폭 학습합니다. 합성곱이 두 개 늘어 CPU에서 30초쯤 걸립니다.

In [ ]:
# 🖐️ 함께 따라하기: Adam 옵티마이저(학습률 0.001)를 만들고 fit_fashion으로 10에폭 학습하세요
res_optimizer = ...
res_history = ...
display(res_history.round(4))

In [ ]:
# 🖐️ 함께 따라하기: 평가 모드로 바꾼 뒤 테스트 데이터로 한 번 평가하세요
...
res_test_loss, res_test_accuracy = ...

# 비교 표에 잔차 블록 CNN 행을 더합니다
compare_rows.append(compare_row("잔차 블록 CNN", res_model, res_history, res_test_accuracy))
display(pd.DataFrame(compare_rows).set_index("모델").round(4))

In [ ]:
# 세 모델의 에폭별 학습 손실을 한 그래프에 그립니다
histories = {"원래 CNN": fashion_history, "배치 정규화 CNN": bn_history, "잔차 블록 CNN": res_history}
plt.figure(figsize=(10, 4.5))
for name in histories:
    plt.plot(histories[name].index, histories[name]["학습 손실"], marker="o", linewidth=2, label=name)
plt.xticks(range(1, 11), fontsize=11)
plt.yticks(fontsize=11)
plt.xlabel("에폭", fontsize=12)
plt.ylabel("학습 손실", fontsize=12)
plt.title("옷 사진 CNN 세 모델의 학습 손실", fontsize=14)
plt.legend(fontsize=12)
plt.show()

**표와 그래프 읽기**
- **학습 손실:** 10에폭 손실이 원래 CNN 0.2848, 배치 정규화 CNN 0.1609, 잔차 블록 CNN 0.1154 순으로 낮고, 그래프에서도 배치 정규화가 든 두 모델의 곡선이 1에폭부터 원래 CNN보다 아래에 있습니다.
- **테스트 정확도 0.8960:** 이번 실행에서 원래 CNN보다 0.0255(51장), 배치 정규화 CNN보다 0.0155(31장) 높았습니다.
- **효과를 나눠 읽기:** 잔차 블록 CNN은 배치 정규화 CNN에 합성곱 두 개(파라미터 4,704개)를 더한 모델이라, 이 차이는 층을 더 쌓은 효과와 잔차 연결의 효과가 합쳐진 것입니다. 잔차 연결만의 효과를 보려면 같은 층에서 지름길만 뺀 모델과 비교해야 하고, 5-4의 논문 실험에서는 그 차이가 18층보다 34층에서 컸습니다.
- **학습과 테스트의 간격:** 잔차 블록 CNN은 학습 데이터 정확도 0.9737과 테스트 0.8960의 간격이 0.078로, 원래 CNN(0.8996과 0.8705, 0.029)보다 큽니다. 학습 손실이 낮다고 처음 보는 사진을 그만큼 더 맞히는 것은 아니므로, 모델은 교안 02에서 검증 데이터로 고릅니다.

## 10. 핵심 정리

- **CNN 입력 모양:** (그림 수, 채널, 높이, 너비)입니다. 흑백은 채널 1, 컬러는 3이며, 흑백 그림 묶음은 `unsqueeze(1)`로 채널 축을 끼웁니다.
- **합성곱:** 필터(이 교안은 3 × 3)를 옮기며 겹친 구역의 값과 같은 자리끼리 곱해 모두 더합니다. 필터의 가중치가 찾는 모양을 정하고(세로선 필터는 세로줄에 6, 가로줄에 0), CNN은 이 가중치를 학습합니다. 이 교안의 합성곱 층은 `nn.Conv2d(입력 채널, 출력 채널, kernel_size=3, padding=1)`입니다.
- **출력 크기와 채널:** 입력 크기 $n$, 필터 크기 $f$, padding $p$, stride 1일 때 출력 크기는 $n - f + 1 + 2p$이고, 3 × 3 필터에 `padding=1`이면 크기가 그대로입니다. 출력 채널 수 = 필터 수 = 특징 맵 수입니다.
- **지역 연결과 파라미터 공유:** 출력 한 칸은 입력의 작은 구역(이 교안은 3 × 3)만 보고(지역 연결), 같은 필터를 모든 자리에서 씁니다(파라미터 공유). (16, 28, 28) → (32, 28, 28)을 `nn.Linear`로 이으면 가중치가 314,703,872개, `nn.Conv2d(16, 32, kernel_size=3)`은 4,608개(절편 포함 4,640개)입니다.
- **최댓값 풀링:** 구역마다 가장 큰 값을 남겨 크기를 줄입니다. 이 교안의 `nn.MaxPool2d(2)`는 2 × 2 구역, stride 2라 가로·세로가 절반입니다(28 → 14 → 7).
- **CNN 쌓기:** [합성곱 → ReLU → 풀링]을 두 번 쌓고 `nn.Flatten()` → `nn.Linear(32 * 7 * 7, 10)`으로 분류합니다. 층마다 모양을 출력해 `nn.Linear`의 입력 크기를 맞춥니다. 마지막 층이 내는 값은 로짓(logit) 10개입니다. 이 교안의 CNN은 파라미터 20,490개로 손글씨 테스트 정확도 0.981이었습니다(지난 시간 DNN은 109,386개, 0.948).
- **수용 영역:** 특징 맵 한 칸이 원래 그림에서 보는 구역입니다. 층이 깊어질수록 넓어지고(3 × 3 합성곱 하나마다 2칸씩, 풀링이 끼면 더 빨리), 이 교안 CNN의 마지막 특징 맵 한 칸은 10 × 10을 봅니다(3 → 4 → 8 → 10). 그래서 뒤 층이 획을 묶은 큰 모양을 찾습니다.
- **기울기 소실:** 역전파가 층마다 곱하는 값의 크기가 1보다 작아 앞쪽 층의 기울기가 0에 가깝게 작아지는 현상입니다. Sigmoid 기울기의 최댓값 0.25를 10번 곱하면 $0.25^{10} \approx 9.54 \times 10^{-7}$이고, 반대로 곱이 커지면 기울기 폭주입니다.
- **잔차 연결:** $y = F(x) + x$로 층 묶음의 입력을 출력에 더합니다. 깊게 쌓을수록 학습 오차가 커지는 성능 저하를 줄이고, $\frac{dy}{dx} = F'(x) + 1$이라 기울기가 지름길로 앞쪽 층까지 전달됩니다(ResNet, 트랜스포머).
- **배치 정규화:** 미니배치마다 채널별로 층 출력을 평균 0, 분산 1로 표준화한 뒤 $\gamma$를 곱하고 $\beta$를 더합니다(`nn.BatchNorm2d(채널 수)`, 합성곱 뒤 활성화 함수 앞). 첫 합성곱 출력의 채널 평균(-0.32~0.29)이 학습 모드에서 0.0, 표준편차 약 1.0으로 맞춰졌습니다. 학습 모드는 미니배치 통계, 평가 모드는 이동 평균을 쓰므로 `model.train()`과 `model.eval()`을 바꿔 부릅니다.
- **잔차 블록과 ResNet:** 이 교안의 잔차 블록(ResNet-18·34의 기본 블록)은 `Conv2d` → `BatchNorm2d` → `ReLU` → `Conv2d` → `BatchNorm2d`의 출력에 입력을 더한 뒤 ReLU를 지나고, 모양이 바뀌면 지름길에 1 × 1 합성곱을 둡니다. `nn.Module`로 만든 채널 16개 블록은 (64, 16, 14, 14)를 같은 모양으로 내보냈습니다. ResNet은 이 블록을 쌓고 전역 평균 풀링과 `Linear`로 끝나며, ImageNet에서 34층 ResNet의 검증 오차(top-1)가 18층보다 낮았습니다(25.03%, 27.88%).
- **학습 결과 읽기:** 학습한 필터, 특징 맵, 틀린 그림, 혼동 행렬(confusion matrix)로 모델이 어떤 특징을 쓰고 어떤 숫자끼리 헷갈리는지 확인합니다. 옷 사진에서는 스웨터, 셔츠, 코트의 맞힌 비율이 약 0.68~0.77로 낮았습니다.
- **저장과 불러오기:** `torch.save(model.state_dict(), 경로)`로 파라미터만 저장하고, 같은 구조로 만든 모델에 `load_state_dict(torch.load(경로, map_location=device))`로 넣습니다.

교안 02에서는 학습 데이터를 줄였을 때 생기는 과적합을 검증 데이터로 찾고, Dropout·weight decay·데이터 증강·조기 종료로 줄인 뒤 같은 학습을 PyTorch Lightning으로 정리합니다.